# Dataset Preparation

Builds the project's evaluation datasets through checkpointed merging, deterministic cleaning, optional LLM rewriting, final validation, and publishing.

## Environment

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from IPython import get_ipython

from pprint import pformat
from typing import Any

formatter = get_ipython().display_formatter.formatters["text/plain"]
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))

## CounterFact

### Raw Snapshot

In [ ]:
# from datasets import load_dataset


# cfact_raw = load_dataset("azhx/counterfact")
# cfact_raw.save_to_disk("../data/datasets/counterfact/raw")

In [ ]:
# from datasets import load_from_disk


# cfact_raw = load_from_disk("../data/datasets/counterfact/raw")
# cfact_raw

### Merge And Target Validation

In [ ]:
# from datasets import concatenate_datasets


# def get_key(record: dict[str, Any]) -> tuple[str, str]:
#     rewrite = record["requested_rewrite"]

#     return (
#         rewrite["subject"],
#         rewrite["relation_id"],
#     )


# cfact_merged = concatenate_datasets([cfact_raw["train"], cfact_raw["test"]])

# target_ids = {}
# for record in cfact_merged:
#     key = get_key(record)
#     rewrite = record["requested_rewrite"]
#     ids = target_ids.setdefault(key, {"true": set(), "new": set()})
#     ids["true"].add(rewrite["target_true"]["id"])
#     ids["new"].add(rewrite["target_new"]["id"])

# valid_keys = {
#     key for key, target_ids in target_ids.items()
#     if len(target_ids["true"]) == 1 and len(target_ids["new"]) == 1
# }
# cfact_merged = cfact_merged.filter(lambda record: get_key(record) in valid_keys)

# cfact_merged.save_to_disk("../data/datasets/counterfact/merged")

In [ ]:
# from datasets import load_from_disk


# cfact_merged = load_from_disk("../data/datasets/counterfact/merged")
# cfact_merged

### Prompt Cleaning

In [ ]:
# import json
# import re
# import unicodedata
# from collections import defaultdict
#
#
# def normalize_prompt(text: str) -> str:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     return " ".join(text.split())
#
#
# id_token_re = re.compile(r"\b(?:ID|PID)\b")
# def is_viable_prompt(prompt: str, answer: str) -> bool:
#     if not normalize_prompt(prompt):
#         return False
#     answer_re = re.compile(rf"\b{re.escape(answer)}\b", re.IGNORECASE)
#     return not answer_re.search(prompt) and\
#            not id_token_re.search(prompt)
#
#
# def get_words(text: str) -> tuple[str, ...]:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     return tuple(re.findall(r"\w+", text))
#
#
# EDIT_SUBJECT_END = "__edit_subject__"
# def build_edit_subject_trie(records) -> dict[str, Any]:
#     edit_subject_trie = {}
#     for record in records:
#         rewrite = record["requested_rewrite"]
#         node = edit_subject_trie
#         for token in get_words(rewrite["subject"]):
#             node = node.setdefault(token, {})
#         node.setdefault(EDIT_SUBJECT_END, set())\
#             .add(rewrite["relation_id"])
#     return edit_subject_trie
#
#
# edit_subject_trie = build_edit_subject_trie(cfact_merged)
# def contains_edit_subject(prompt: str, relation_id: str) -> bool:
#     words = get_words(prompt)
#     for start in range(len(words)):
#         node = edit_subject_trie
#         for end in range(start, len(words)):
#             node = node.get(words[end])
#             if node is None:
#                 break
#             relations = node.get(EDIT_SUBJECT_END, set())
#             if relation_id in relations:
#                 return True
#     return False
#
#
# def build_relation_templates(records) -> dict[str, tuple[tuple[str, str], ...]]:
#     templates = defaultdict(set)
#     for record in records:
#         rewrite = record["requested_rewrite"]
#         template = rewrite["prompt"]
#         if template.count("{}") != 1:
#             raise ValueError(f"Unexpected CounterFact template: {template}")
#         templates[rewrite["relation_id"]].add(tuple(template.split("{}")))
#     return {
#         relation_id: tuple(sorted(
#             values,
#             key=lambda value: len(value[0]) + len(value[1]),
#             reverse=True,
#         ))
#         for relation_id, values in templates.items()
#     }
#
#
# relation_templates = build_relation_templates(cfact_merged)
# def extract_relation_subject(prompt: str, relation_id: str) -> str:
#     matches = []
#     for prefix, suffix in relation_templates[relation_id]:
#         if not prompt.startswith(prefix) or not prompt.endswith(suffix):
#             continue
#         end = len(prompt) - len(suffix) if suffix else len(prompt)
#         subject = prompt[len(prefix):end]
#         if subject:
#             matches.append((len(prefix) + len(suffix), subject))
#     if not matches:
#         raise ValueError(f"No CounterFact template matched: {prompt}")
#     longest = max(length for length, _ in matches)
#     subjects = {
#         normalize_prompt(subject): subject
#         for length, subject in matches
#         if length == longest
#     }
#     if len(subjects) != 1:
#         raise ValueError(f"Ambiguous CounterFact locality subject: {prompt}")
#     return next(iter(subjects.values()))
#
#
# def render_edit_prompt(record: dict[str, Any]) -> dict[str, Any]:
#     rewrite = dict(record["requested_rewrite"])
#     rewrite["prompt"] = rewrite["prompt"].format(rewrite["subject"])
#     return {"requested_rewrite": rewrite}
#
#
# def clean_record(record: dict[str, Any]) -> dict[str, Any]:
#     rewrite = record["requested_rewrite"]
#     relation_id = rewrite["relation_id"]
#     true_answer = rewrite["target_true"]["str"]
#     new_answer = rewrite["target_new"]["str"]
#
#     seen_prompts = {normalize_prompt(rewrite["prompt"])}
#
#     def clean_prompts(prompts: list[str], answer: str, locality: bool = False) -> list[str]:
#         cleaned = []
#         for prompt in prompts:
#             prompt_key = normalize_prompt(prompt)
#             if prompt_key in seen_prompts:
#                 continue
#             seen_prompts.add(prompt_key)
#             if not is_viable_prompt(prompt, answer):
#                 continue
#             if locality and contains_edit_subject(prompt, relation_id):
#                 continue
#             cleaned.append(prompt)
#         return cleaned
#
#     generation = clean_prompts(record["generation_prompts"], new_answer)
#     neighborhood = clean_prompts(record["neighborhood_prompts"], true_answer, True)
#     attribute = clean_prompts(record["attribute_prompts"], new_answer, True)
#
#     return {
#         "neighborhood_prompts": neighborhood,
#         "attribute_prompts": attribute,
#         "generation_prompts": generation,
#     }
#
#
# def find_conflicting_locality_facts(records) -> set[tuple[str, str]]:
#     answers = defaultdict(set)
#     for record in records:
#         rewrite = record["requested_rewrite"]
#         relation_id = rewrite["relation_id"]
#         for field, answer_field in (
#             ("neighborhood_prompts", "target_true"),
#             ("attribute_prompts", "target_new"),
#         ):
#             answer_id = rewrite[answer_field]["id"]
#             for prompt in record[field]:
#                 subject = extract_relation_subject(prompt, relation_id)
#                 answers[(normalize_prompt(subject), relation_id)].add(answer_id)
#     return {
#         key for key, answer_ids in answers.items()
#         if len(answer_ids) > 1
#     }
#
#
# def remove_conflicting_locality(record: dict[str, Any]) -> dict[str, Any]:
#     relation_id = record["requested_rewrite"]["relation_id"]
#
#     def keep(prompt: str) -> bool:
#         subject = extract_relation_subject(prompt, relation_id)
#         return (normalize_prompt(subject), relation_id) not in conflicting_locality_facts
#
#     return {
#         "neighborhood_prompts": [
#             prompt for prompt in record["neighborhood_prompts"] if keep(prompt)
#         ],
#         "attribute_prompts": [
#             prompt for prompt in record["attribute_prompts"] if keep(prompt)
#         ],
#     }
#
#
# def add_locality_subjects(record: dict[str, Any]) -> dict[str, Any]:
#     relation_id = record["requested_rewrite"]["relation_id"]
#     return {
#         "neighborhood_subjects": [
#             extract_relation_subject(prompt, relation_id)
#             for prompt in record["neighborhood_prompts"]
#         ],
#         "attribute_subjects": [
#             extract_relation_subject(prompt, relation_id)
#             for prompt in record["attribute_prompts"]
#         ],
#     }
#
#
# cfact_clean = cfact_merged\
#     .remove_columns(["case_id", "pararel_idx", "paraphrase_prompts"])\
#     .map(render_edit_prompt)\
#     .filter(lambda record: is_viable_prompt(
#         record["requested_rewrite"]["prompt"],
#         record["requested_rewrite"]["target_new"]["str"]
#     ))\
#     .map(clean_record)
#
# locality_before = sum(
#     len(record["neighborhood_prompts"]) + len(record["attribute_prompts"])
#     for record in cfact_clean
# )
# conflicting_locality_facts = find_conflicting_locality_facts(cfact_clean)
# cfact_clean = cfact_clean\
#     .map(remove_conflicting_locality)\
#     .map(add_locality_subjects)
# locality_after = sum(
#     len(record["neighborhood_prompts"]) + len(record["attribute_prompts"])
#     for record in cfact_clean
# )
# cfact_clean_report = {
#     "rows": len(cfact_clean),
#     "conflicting_locality_facts": len(conflicting_locality_facts),
#     "locality_before": locality_before,
#     "locality_removed": locality_before - locality_after,
#     "locality_after": locality_after,
# }
#
# cfact_clean_path = "../data/datasets/counterfact/clean"
# cfact_clean.save_to_disk(cfact_clean_path)
# with open(f"{cfact_clean_path}/report.json", "w", encoding="utf-8") as file:
#     json.dump(cfact_clean_report, file, ensure_ascii=False, indent=2)
# cfact_clean_report

In [ ]:
# from datasets import load_from_disk


# cfact_clean = load_from_disk("../data/datasets/counterfact/clean")
# cfact_clean

### Evaluation Sampling

In [ ]:
# import unicodedata
# from random import Random


# SEED = 42
# N_GENERATION = 3
# N_NEIGHBORHOOD = 2
# N_ATTRIBUTE = 2


# def sample_prompts(prompts: list[str], n: int, seed_key: str) -> list[str]:
#     prompts = list(prompts)
#     Random(f"{SEED}:{seed_key}").shuffle(prompts)
#     return prompts[:n]
#
#
# def locality_subject_key(subject: str) -> str:
#     subject = unicodedata.normalize("NFKC", str(subject)).casefold()
#     return " ".join(subject.split())
#
#
# def count_locality_facts(subjects: list[str]) -> int:
#     return len({locality_subject_key(subject) for subject in subjects})
#
#
# def sample_locality(
#     prompts: list[str],
#     subjects: list[str],
#     n: int,
#     seed_key: str,
# ) -> list[tuple[str, str]]:
#     candidates = list(zip(prompts, subjects))
#     Random(f"{SEED}:{seed_key}").shuffle(candidates)
#
#     selected = []
#     seen_subjects = set()
#     for prompt, subject in candidates:
#         subject_key = locality_subject_key(subject)
#         if subject_key in seen_subjects:
#             continue
#         seen_subjects.add(subject_key)
#         selected.append((prompt, subject))
#         if len(selected) == n:
#             break
#     return selected
#
#
# def has_uniform_candidates(record: dict[str, Any]) -> bool:
#     return (
#         len(record["generation_prompts"]) >= N_GENERATION and
#         count_locality_facts(record["neighborhood_subjects"]) >= N_NEIGHBORHOOD and
#         count_locality_facts(record["attribute_subjects"]) >= N_ATTRIBUTE
#     )


# def uniform(record: dict[str, Any], idx: int) -> dict[str, Any]:
#     rewrite = record["requested_rewrite"]

#     true_answer = rewrite["target_true"]
#     new_answer = rewrite["target_new"]

#     generation = sample_prompts(
#         record["generation_prompts"],
#         N_GENERATION,
#         f"{idx}:generation",
#     )

#     neighborhood = sample_locality(
#         record["neighborhood_prompts"],
#         record["neighborhood_subjects"],
#         N_NEIGHBORHOOD,
#         f"{idx}:neighborhood",
#     )

#     attribute = sample_locality(
#         record["attribute_prompts"],
#         record["attribute_subjects"],
#         N_ATTRIBUTE,
#         f"{idx}:attribute",
#     )

#     return {
#         "edit": {
#             "prompt": rewrite["prompt"],
#             "relation_id": rewrite["relation_id"],
#             "subject": rewrite["subject"],
#             "answer": {
#                 "true": true_answer,
#                 "new": new_answer,
#             },
#         },
#         "generality": generation,
#         "locality": [
#             {
#                 "prompt": prompt,
#                 "subject": subject,
#                 "answer": true_answer,
#             }
#             for prompt, subject in neighborhood
#         ] + [
#             {
#                 "prompt": prompt,
#                 "subject": subject,
#                 "answer": new_answer,
#             }
#             for prompt, subject in attribute
#         ],
#     }


# cfact_uniform = cfact_clean\
#     .filter(has_uniform_candidates)\
#     .map(
#         uniform,
#         with_indices=True,
#         remove_columns=cfact_clean.column_names
#     )

# cfact_uniform.save_to_disk("../data/datasets/counterfact/uniform")

In [ ]:
# from datasets import load_from_disk
#
#
# cfact_uniform = load_from_disk("../data/datasets/counterfact/uniform")
# cfact_uniform

### Prompt Enhancement

In [ ]:
# import asyncio
# import json
# import re
# import unicodedata
#
# from datasets import Dataset
# from openai import AsyncOpenAI
# from typing import Any
# from tqdm.auto import tqdm
#
#
# def normalize_prompt(text: str) -> str:
#     text = unicodedata.normalize("NFKC", str(text))
#     return " ".join(text.split())
#
#
# def prompt_key(text: str) -> str:
#     return normalize_prompt(text).casefold()
#
#
# def contains_exact_text(prompt: str, text: str, ignore_case: bool = False) -> bool:
#     prompt = normalize_prompt(prompt)
#     text = normalize_prompt(text)
#     if not text:
#         return False
#     flags = re.IGNORECASE if ignore_case else 0
#     return re.search(rf"(?<!\w){re.escape(text)}(?!\w)", prompt, flags) is not None
#
#
# def answer_leaks_into_prompt(prompt: str, answer: str) -> bool:
#     return contains_exact_text(prompt, answer, ignore_case=True)
#
#
# def strip_json_response(text: str) -> str:
#     text = text.strip().lstrip("\ufeff")
#     fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, re.IGNORECASE | re.DOTALL)
#     return fenced.group(1).strip() if fenced else text
#
#
# def to_enhancement_record(record: dict[str, Any]) -> dict[str, Any]:
#     return {
#         "edit": {
#             "prompt": record["edit"]["prompt"],
#             "subject": record["edit"]["subject"],
#             "answer": {
#                 "true": record["edit"]["answer"]["true"]["str"],
#                 "new": record["edit"]["answer"]["new"]["str"],
#             },
#         },
#         "generality": list(record["generality"]),
#         "locality_subjects": [
#             item["subject"] for item in record["locality"]
#         ],
#         "locality": [
#             {"prompt": item["prompt"], "answer": item["answer"]["str"]}
#             for item in record["locality"]
#         ],
#     }
#
#
# def restore_enhanced_record(source: dict[str, Any], enhanced: dict[str, Any]) -> dict[str, Any]:
#     return {
#         "edit": {
#             "prompt": normalize_prompt(enhanced["edit"]["prompt"]),
#             "relation_id": source["edit"]["relation_id"],
#             "subject": source["edit"]["subject"],
#             "answer": source["edit"]["answer"],
#         },
#         "generality": [
#             normalize_prompt(prompt) for prompt in enhanced["generality"]
#         ],
#         "locality": [
#             {
#                 "prompt": normalize_prompt(output["prompt"]),
#                 "subject": item["subject"],
#                 "answer": item["answer"],
#             }
#             for item, output in zip(source["locality"], enhanced["locality"])
#         ],
#     }
#
#
# def get_enhancement_examples() -> list[dict[str, Any]]:
#     return [
#         {
#             "purpose": "Repair the entire record. Every weak edit, generality, and locality prompt is improved while preserving the mother-tongue relation.",
#             "input": {
#                 "edit": {
#                     "prompt": "Clara Voss spoke the language",
#                     "subject": "Clara Voss",
#                     "answer": {"true": "German", "new": "Spanish"},
#                 },
#                 "generality": [
#                     "Where Clara Voss was born, people speak the language of",
#                     "Clara Voss was born in",
#                     "Clara Voss, a native",
#                 ],
#                 "locality_subjects": ["Martin Keller", "Elise Bauer", "Rafael Moreno", "Lucia Torres"],
#                 "locality": [
#                     {"prompt": "Martin Keller, speaker of", "answer": "German"},
#                     {"prompt": "Elise Bauer spoke the language", "answer": "German"},
#                     {"prompt": "Rafael Moreno, a native", "answer": "Spanish"},
#                     {"prompt": "Lucia Torres spoke the language", "answer": "Spanish"},
#                 ],
#             },
#             "output": {
#                 "edit": {
#                     "prompt": "Clara Voss's mother tongue is",
#                     "subject": "Clara Voss",
#                     "answer": {"true": "German", "new": "Spanish"},
#                 },
#                 "generality": [
#                     "Clara Voss's native language is",
#                     "The first language Clara Voss learned was",
#                     "The language Clara Voss first spoke was",
#                 ],
#                 "locality_subjects": ["Martin Keller", "Elise Bauer", "Rafael Moreno", "Lucia Torres"],
#                 "locality": [
#                     {"prompt": "Martin Keller's native language is", "answer": "German"},
#                     {"prompt": "The mother tongue of Elise Bauer is", "answer": "German"},
#                     {"prompt": "Rafael Moreno's native language is", "answer": "Spanish"},
#                     {"prompt": "Lucia Torres's mother tongue is", "answer": "Spanish"},
#                 ],
#             },
#         },
#         {
#             "purpose": "Repair under-directed fragments across edit, generality, and locality while preserving cloze type and instrument relation.",
#             "input": {
#                 "edit": {
#                     "prompt": "Nolan Reed, the",
#                     "subject": "Nolan Reed",
#                     "answer": {"true": "guitar", "new": "piano"},
#                 },
#                 "generality": [
#                     "Nolan Reed plays",
#                     "Nolan Reed is known for",
#                     "Nolan Reed's sound is based around the",
#                 ],
#                 "locality_subjects": ["Ari Lane", "Sofia Vale", "Mara Chen", "Kai Soto"],
#                 "locality": [
#                     {"prompt": "Ari Lane, the", "answer": "guitar"},
#                     {"prompt": "Sofia Vale's music uses the", "answer": "guitar"},
#                     {"prompt": "Mara Chen, the", "answer": "piano"},
#                     {"prompt": "Kai Soto performs", "answer": "piano"},
#                 ],
#             },
#             "output": {
#                 "edit": {
#                     "prompt": "Nolan Reed's primary instrument is the",
#                     "subject": "Nolan Reed",
#                     "answer": {"true": "guitar", "new": "piano"},
#                 },
#                 "generality": [
#                     "The instrument Nolan Reed plays is the",
#                     "Nolan Reed is known for playing the",
#                     "Nolan Reed's main instrument is the",
#                 ],
#                 "locality_subjects": ["Ari Lane", "Sofia Vale", "Mara Chen", "Kai Soto"],
#                 "locality": [
#                     {"prompt": "Ari Lane's primary instrument is the", "answer": "guitar"},
#                     {"prompt": "Sofia Vale's main instrument is the", "answer": "guitar"},
#                     {"prompt": "Mara Chen's primary instrument is the", "answer": "piano"},
#                     {"prompt": "The instrument Kai Soto performs on is the", "answer": "piano"},
#                 ],
#             },
#         },
#         {
#             "purpose": "Repair all prompts while keeping citizenship as legal citizenship, not residence, birthplace, passport ownership, ethnicity, or workplace.",
#             "input": {
#                 "edit": {
#                     "prompt": "Mira Solberg is from",
#                     "subject": "Mira Solberg",
#                     "answer": {"true": "Norway", "new": "Belgium"},
#                 },
#                 "generality": [
#                     "Mira Solberg currently lives in",
#                     "The passport that Mira Solberg carries is from",
#                     "Mira Solberg belongs to the country of",
#                 ],
#                 "locality_subjects": ["Jonas Eide", "Ingrid Dahl", "Elise Laurent", "René Martel"],
#                 "locality": [
#                     {"prompt": "Jonas Eide is from", "answer": "Norway"},
#                     {"prompt": "Ingrid Dahl belongs to the country of", "answer": "Norway"},
#                     {"prompt": "Elise Laurent lives in", "answer": "Belgium"},
#                     {"prompt": "René Martel has papers from", "answer": "Belgium"},
#                 ],
#             },
#             "output": {
#                 "edit": {
#                     "prompt": "Mira Solberg holds citizenship from",
#                     "subject": "Mira Solberg",
#                     "answer": {"true": "Norway", "new": "Belgium"},
#                 },
#                 "generality": [
#                     "Mira Solberg is a citizen of",
#                     "The country of citizenship of Mira Solberg is",
#                     "Mira Solberg legally holds citizenship from",
#                 ],
#                 "locality_subjects": ["Jonas Eide", "Ingrid Dahl", "Elise Laurent", "René Martel"],
#                 "locality": [
#                     {"prompt": "Jonas Eide holds citizenship from", "answer": "Norway"},
#                     {"prompt": "The country of citizenship of Ingrid Dahl is", "answer": "Norway"},
#                     {"prompt": "Elise Laurent is a citizen of", "answer": "Belgium"},
#                     {"prompt": "The citizenship held by René Martel is from", "answer": "Belgium"},
#                 ],
#             },
#         },
#         {
#             "purpose": "Fail instead of cosmetically rewriting duplicate locality facts. The first two locality prompts ask the same subject-relation-answer fact.",
#             "input": {
#                 "edit": {
#                     "prompt": "The instrument Nolan Reed plays is the",
#                     "subject": "Nolan Reed",
#                     "answer": {"true": "guitar", "new": "piano"},
#                 },
#                 "generality": [
#                     "Nolan Reed is known for playing the",
#                     "Nolan Reed performs on the",
#                     "Nolan Reed's main instrument is the",
#                 ],
#                 "locality_subjects": ["Ari Lane", "Ari Lane", "Mara Chen", "Kai Soto"],
#                 "locality": [
#                     {"prompt": "Ari Lane plays the", "answer": "guitar"},
#                     {"prompt": "Ari Lane performs on the", "answer": "guitar"},
#                     {"prompt": "Mara Chen plays the", "answer": "piano"},
#                     {"prompt": "The instrument Kai Soto performs on is the", "answer": "piano"},
#                 ],
#             },
#             "output": {"status": "failed", "reason": "locality contains repeated subject-relation-answer facts"},
#         },
#     ]
#
#
# CFACT_ENHANCEMENT_EXAMPLES = json.dumps(
#     get_enhancement_examples(), ensure_ascii=False, indent=2
# )
#
#
# CFACT_RELATION_GUIDANCE = {
#     "P19": "place of birth; every prompt must explicitly ask where the subject was born or for its birthplace, never origin, residence, parents, nationality, or regional language",
#     "P37": "an officially recognized language; do not reduce it to a language merely spoken, understood, or commonly used, and do not imply uniqueness",
#     "P101": "field, domain, or focus of work; use answer-compatible expertise, activity, scholarly, professional, or creative-focus wording when literal field wording is unnatural, without drifting to employer, work location, or formal position",
#     "P103": "native language, mother tongue, or first language; never infer it from birthplace, nationality, origin, or any language the subject merely speaks or writes",
#     "P106": "occupation or profession; use article-free wording because the prompt must fit both answers, and do not change it into field of work, employer, position held, accomplishment, or known-for wording",
#     "P1412": "a language the subject speaks, writes, signs, or otherwise uses; prefer modality-neutral wording unless the modality is supported, and do not change it into native language, birthplace, residence, nationality, or friends' language",
#     "P176": "manufacturer; ask directly who manufactured or produced the subject, not who created, developed, sold, owned, or supervised production of it",
#     "P178": "developer; ask directly who developed the subject, not its manufacturer, owner, distributor, seller, or generic creator",
#     "P276": "physical location; ask directly where the subject is located, preserving available geographic granularity and avoiding navigation, surroundings, restaurants, headquarters, country, or administrative-containment drift",
#     "P131": "administrative territorial entity containing the subject; explicitly ask which administrative territory contains it, never nearby places, restaurants, surroundings, navigation, origin, or headquarters",
#     "P495": "country of origin; ask directly for the country of origin and avoid medium-specific created, developed, or formulated wording unless natural for the subject; never ask regional language, formation location, publication place, nationality, or headquarters",
#     "P740": "formation or founding location; use formed, founded, or established for organizations and groups, but use the closest answer-compatible origin wording for subjects that cannot naturally be founded; never invent founders or inspiration",
#     "P937": "place where the subject works or is professionally based; ask directly for work location, not employer, residence, birthplace, commute, nearby surroundings, office meals, or lunch",
#     "P1303": "musical instrument, voice, or performance technique used by the subject; choose answer-compatible plays, sings, uses, or neutral performance-medium wording, never genre, occupation, vague known-for/skill wording, subjective praise, or incomplete apposition",
#     "P17": "country containing or associated with the subject; explicitly ask which country, never a city, region, birthplace, origin, navigation route, surroundings, or restaurants",
#     "P20": "place of death; ask directly where the subject died or passed away, without inventing that they were killed or changing it into burial place, final residence, local events, or regional language",
#     "P27": "country of citizenship; prefer citizen of or holds citizenship in, without implying uniqueness or changing it into birthplace, residence, ethnicity, origin, passport ownership, or native language",
#     "P30": "continent; explicitly ask which continent contains or is associated with the subject, never country, region, navigation, surroundings, or regional language",
#     "P36": "capital city; ask directly for the subject's capital, never headquarters, administrative office, tourist attractions, reputation, or language spoken there",
#     "P39": "formal position, office, title, or appointed role held; prefer holds the office/title of or serves as, and do not reduce it to a general field, accomplishment, or playing position",
#     "P108": "employer or employing organization; use employed by, works for, or professionally affiliated with, never a bare of fragment, work location, occupation, or unsupported narrative about work",
#     "P127": "owner; explicitly ask who or what owns the subject, never vague by/from wording, seller, trademark owner, manufacturer, developer, creator, or member organization",
#     "P136": "genre or the closest medium-specific classification; choose wording that fits the subject and answer, such as musical style or type of music, literary genre or type of book, screen genre, theatrical form, or a neutral type/style/category when the medium is unclear. Do not force the literal word genre or reuse templates based on play, perform, greatest artistic work, or incredible at. Preserve the classification meaning as closely as possible without drifting to instrument, occupation, creator, or a specific work",
#     "P138": "name source or namesake; ask what person, place, entity, material, or concept the subject was named after, without assuming commemoration or using known-for and incomplete reason clauses",
#     "P140": "religion, faith, religious identity, or tradition associated with the subject; use neutral wording that fits people and institutions and noun or adjective answers, without introducing official status, worship, attendance, values, ethnicity, or nationality",
#     "P159": "headquarters location; ask directly where the subject is headquartered and use the plural noun headquarters, never navigation, surroundings, restaurants, general location, origin, or formation location",
#     "P190": "twinned administrative counterpart; ask directly which city, town, region, or administrative body the subject is twinned with, never the counterpart's language, attractions, reputation, or another attribute",
#     "P264": "record label associated with the subject; distinguish a performer signed to or represented by a label from a work released on or by a label, using neutral associated-label wording when uncertain; never ownership, sales, contracts, broadcaster, or manufacturer",
#     "P364": "original language of an audiovisual work; preserve the original qualifier and use originally made or produced in when natural, never treat the title as a place or ask production country, filming location, inhabitants' language, or creator language",
#     "P407": "language used by the work or name; use written in only for clearly textual subjects and otherwise prefer neutral language used in/by, without inferring author origin, country, creator language, or originality",
#     "P413": "sport playing position or on-field role; ask directly which position the athlete plays using sport-neutral wording, never expertise, strength, occupation, formal position held, activity, or incomplete apposition",
#     "P449": "original network, channel, broadcaster, or distribution platform on which the subject first aired or premiered; make that role explicit and never ask for another program, add subjective opinions, or change it into production company or release location",
#     "P463": "membership in a group, organization, alliance, federation, team, or band; prefer direct member-of wording and never weaken it into generic affiliation, ownership, employment, mereological part-of, or known-for wording",
# }
#
#
# def build_enhancement_prompt(record: dict[str, Any], relation_id: str) -> str:
#     payload = json.dumps(record, ensure_ascii=False, indent=2)
#     relation_guidance = CFACT_RELATION_GUIDANCE.get(
#         relation_id,
#         f"Wikidata relation {relation_id}; preserve its exact meaning when natural, otherwise use the closest answer-compatible paraphrase",
#     )
#     return (
#         "Repair one CounterFact evaluation record.\n"
#         "Only edit prompt strings. All answers, subjects, list lengths, and list order are locked.\n"
#         f"Locked relation for every prompt in this record: {relation_guidance}.\n"
#         "The relation identifier is context only. Do not add it to the output.\n"
#         "You must inspect and repair the whole record: edit.prompt, every generality prompt, and every locality prompt.\n"
#         "Do not assume only generality needs rewriting. If a locality or edit prompt is noisy, under-directed, or relation-drifted, repair it too.\n"
#         "Leave prompts unchanged when they are already clear and correct; otherwise make the smallest safe repair.\n"
#         "The input generality strings come from CounterFact generation prompts and are not trusted paraphrases. Treat their wording as raw material, not semantics that must be preserved.\n"
#         "Rewrite every generality string into a genuine paraphrase of edit.prompt: it must query the exact same subject-relation fact, keep the exact edit subject, and make edit.answer.new the direct answer.\n"
#         "Generality prompts must differ meaningfully in wording, but must not test residence, origin, nearby places, skills, ownership, or any other neighboring fact unless that is the locked edit relation.\n"
#         "locality_subjects gives the exact subject of each locality item in matching order. Return it unchanged, and keep each subject verbatim in its prompt.\n"
#         "Locality is a list of prompt-answer objects. Every locality prompt uses the same locked relation but keeps its own subject and is judged only against its own assigned answer.\n"
#         "For a successful repair, return one JSON object with exactly the same schema as the input. "
#         "If the record cannot be repaired without answer leakage, relation drift, or semantic duplicate locality facts, "
#         "return exactly {\"status\": \"failed\", \"reason\": \"short reason\"}.\n\n"
#         "Central repair goal:\n"
#         "Each prompt must make its assigned answer the direct, natural, preferred answer. "
#         "Repair noisy fragments and relation drift. Prefer the exact intended fact, but when literal relation wording "
#         "cannot form a natural prompt for the subject-answer pair, use the closest logically equivalent wording. "
#         "The locked record relation defines the intended fact type for edit, generality, and locality. Locality differs only in subject and assigned answer, not relation.\n\n"
#         "Relation discipline:\n"
#         "- Distinguish the intended relation from nearby concepts such as birthplace versus nationality, creator versus owner, or instrument versus occupation.\n"
#         "- Preserve exact relation meaning when it can be expressed naturally. If it cannot, use the nearest answer-compatible synonym, category, or medium-specific wording.\n"
#         "- For generality, the locked edit relation always overrides the input prompt's apparent relation. Replace conflicting predicates instead of preserving their meaning.\n"
#         "- If an edit or locality prompt drifts to a nearby relation, repair it toward the intended meaning without changing its entities or answer.\n"
#         "- Apply the locked relation to locality as well. Repair ambiguous or drifting locality wording instead of trusting its apparent predicate.\n\n"
#         "Rules:\n"
#         "1) Do not change any answer string, including edit.answer.true, edit.answer.new, or locality[*].answer.\n"
#         "2) Do not change edit.subject. Keep the exact full subject with the same casing in edit.prompt and every generality prompt.\n"
#         "3) Do not add, remove, replace, or merge named entities in any prompt.\n"
#         "4) You may change between cloze and question form when that produces a clearer, more natural prompt.\n"
#         "5) Keep cloze prompts as natural prefixes ending immediately before the answer. Do not use blanks or placeholders.\n"
#         "6) Keep question prompts as direct questions ending with a question mark.\n"
#         "7) Do not include meta-instructions such as 'answer with', 'respond with', 'in one word', or output-format instructions.\n"
#         "8) The assigned evaluation answer must not appear verbatim inside its prompt. For edit.prompt and generality prompts, only edit.answer.new is forbidden because those prompts are judged after editing. For locality prompts, locality[*].answer is forbidden.\n"
#         "9) Avoid duplicate or near-duplicate wording within the whole record, including edit, generality, and locality. "
#         "Generality prompts intentionally test the edited fact, but their wording must still be meaningfully different.\n"
#         "10) Within locality, if two prompts query the same subject-relation-answer fact, return failed instead of saving a cosmetic paraphrase.\n"
#         "11) Prefer concise natural wording over noisy sentence fragments.\n"
#         "12) Reject the record when an answer is questionable, wrong-type, ambiguous, or cannot be made the direct answer without changing the underlying fact.\n\n"
#         "Synthetic full-record examples. Follow the behavior and schema, not the names or facts. "
#         "Notice that successful examples repair edit, generality, and locality prompts when they are weak.\n"
#         f"{CFACT_ENHANCEMENT_EXAMPLES}\n\n"
#         "Input JSON:\n"
#         f"{payload}\n\n"
#         "Output JSON only:"
#     )
#
#
# def validate_enhanced_record(source: dict[str, Any], enhanced) -> list[str]:
#     errors: list[str] = []
#
#     if not isinstance(enhanced, dict):
#         return ["output is not a JSON object"]
#
#     if set(enhanced) != {"edit", "generality", "locality_subjects", "locality"}:
#         return ["output keys must be exactly: edit, generality, locality_subjects, locality"]
#     if not isinstance(enhanced["edit"], dict):
#         return ["edit is not an object"]
#     if set(enhanced["edit"]) != {"prompt", "subject", "answer"}:
#         return ["edit keys must be exactly: prompt, subject, answer"]
#     if not isinstance(enhanced["generality"], list):
#         return ["generality is not a list"]
#     if not isinstance(enhanced["locality_subjects"], list):
#         return ["locality_subjects is not a list"]
#     if not isinstance(enhanced["locality"], list):
#         return ["locality is not a list"]
#
#     if enhanced["edit"].get("subject") != source["edit"].get("subject"):
#         errors.append("edit.subject changed")
#     if enhanced["edit"].get("answer") != source["edit"].get("answer"):
#         errors.append("edit.answer changed")
#     if not isinstance(enhanced["edit"].get("prompt"), str):
#         errors.append("edit.prompt is not a string")
#
#     subject = source["edit"].get("subject", "")
#     edit_prompt = enhanced["edit"].get("prompt", "")
#     if subject and not contains_exact_text(edit_prompt, subject):
#         errors.append("edit.prompt missing exact subject")
#
#     if len(enhanced.get("generality", [])) != len(source.get("generality", [])):
#         errors.append("generality length changed")
#     else:
#         for idx, out_prompt in enumerate(enhanced["generality"]):
#             if not isinstance(out_prompt, str):
#                 errors.append(f"generality[{idx}] is not a prompt string")
#             elif subject and not contains_exact_text(out_prompt, subject):
#                 errors.append(f"generality[{idx}] missing exact subject")
#
#     locality_subjects = source.get("locality_subjects", [])
#     if enhanced.get("locality_subjects") != locality_subjects:
#         errors.append("locality_subjects changed")
#     if len(locality_subjects) != len(source.get("locality", [])):
#         errors.append("locality_subjects length does not match locality")
#
#     if len(enhanced.get("locality", [])) != len(source.get("locality", [])):
#         errors.append("locality length changed")
#     else:
#         for idx, (src_item, out_item) in enumerate(zip(source["locality"], enhanced["locality"])):
#             if not isinstance(out_item, dict):
#                 errors.append(f"locality[{idx}] is not a prompt-answer object")
#             elif set(out_item) != {"prompt", "answer"}:
#                 errors.append(f"locality[{idx}] keys must be exactly: prompt, answer")
#             elif not isinstance(out_item.get("prompt"), str):
#                 errors.append(f"locality[{idx}].prompt is not a string")
#             elif out_item.get("answer") != src_item.get("answer"):
#                 errors.append(f"locality[{idx}].answer changed")
#             elif idx < len(locality_subjects) and not contains_exact_text(
#                 out_item["prompt"], locality_subjects[idx]
#             ):
#                 errors.append(f"locality[{idx}].prompt missing exact subject")
#
#     if errors:
#         return errors
#
#     prompt_pairs = [("edit", enhanced["edit"].get("prompt", ""), [source["edit"]["answer"]["new"]])]
#     prompt_pairs.extend(
#         (f"generality[{idx}]", out, [source["edit"]["answer"]["new"]])
#         for idx, out in enumerate(enhanced["generality"])
#         if isinstance(out, str)
#     )
#     prompt_pairs.extend(
#         (f"locality[{idx}]", out["prompt"], [src["answer"]])
#         for idx, (src, out) in enumerate(zip(source["locality"], enhanced["locality"]))
#         if isinstance(src, dict) and isinstance(out, dict)
#     )
#
#     seen: dict[str, str] = {}
#     for label, out_prompt, forbidden_answers in prompt_pairs:
#         prompt = normalize_prompt(out_prompt)
#         if not prompt:
#             errors.append(f"{label}.prompt is empty")
#             continue
#
#         lowered = prompt.casefold()
#         if any(token in lowered for token in (
#             "{}", "____", "[blank]", "<blank>", "[mask]",
#             "answer with", "respond with", "in one word",
#         )):
#             errors.append(f"{label}.prompt contains forbidden formatting/meta text")
#         if re.search(r"\b(?:ID|PID)\b", prompt):
#             errors.append(f"{label}.prompt contains ID/PID")
#
#         leaked_answers = [answer for answer in forbidden_answers if answer_leaks_into_prompt(prompt, answer)]
#         if leaked_answers:
#             errors.append(f"{label}.prompt leaks answer(s): {leaked_answers}")
#
#         key = prompt_key(prompt)
#         if key in seen:
#             errors.append(f"duplicate prompt: {seen[key]} and {label}")
#         else:
#             seen[key] = label
#
#     return errors
#
#
# async def enhance_record(
#     record: dict[str, Any],
#     client: AsyncOpenAI,
#     config: dict[str, Any],
#     feedback: list[str] | None = None,
# ) -> tuple[dict[str, Any] | None, list[str], Any]:
#     request_record = to_enhancement_record(record)
#     messages = [
#         {
#             "role": "system",
#             "content": "You repair benchmark prompts. You only change prompt text and never change answers.",
#         },
#         {
#             "role": "user",
#             "content": build_enhancement_prompt(
#                 request_record, record["edit"]["relation_id"]
#             ) + (
#                 "\n\nPrevious attempt failed validation. Fix every issue below and return only corrected JSON:\n"
#                 + "\n".join(f"- {error}" for error in feedback)
#                 if feedback else ""
#             ),
#         },
#     ]
#
#     completion = await client.chat.completions.create(
#         model=config["model"],
#         messages=messages,
#         **config.get("request_config", {}),
#     )
#
#     content = completion.choices[0].message.content
#     if not isinstance(content, str) or not content.strip():
#         return None, ["empty model response"], None
#
#     response_text = strip_json_response(content)
#     try:
#         enhanced = json.loads(response_text)
#     except Exception as exc:
#         return None, [f"invalid JSON: {exc}"], {"raw_response": response_text}
#
#     if isinstance(enhanced, dict) and enhanced.get("status") == "failed":
#         reason = enhanced.get("reason", "no reason")
#         return None, [f"model returned failed status: {reason}"], enhanced
#
#     errors = validate_enhanced_record(request_record, enhanced)
#     if errors:
#         return None, errors, enhanced
#
#     return restore_enhanced_record(record, enhanced), [], None
#
#
# async def enhance_dataset(
#     dataset,
#     config: dict[str, Any],
#     start: int = 0,
#     limit: int | None = None,
# ) -> tuple[Dataset, list[dict[str, Any]]]:
#     if start < 0:
#         raise ValueError("start must be non-negative")
#     if limit is not None and limit < 0:
#         raise ValueError("limit must be non-negative or None")
#
#     end = len(dataset) if limit is None else min(len(dataset), start + limit)
#     indices = list(range(start, end))
#
#     max_concurrency = config.get("max_concurrency", 4)
#     max_retries = config.get("max_retries", 3)
#     retry_delay = config.get("retry_delay", 2.0)
#     if max_concurrency <= 0:
#         raise ValueError("max_concurrency must be positive")
#     if max_retries < 0 or retry_delay < 0:
#         raise ValueError("max_retries and retry_delay must be non-negative")
#     if not indices:
#         return dataset.select([]), []
#
#     client = AsyncOpenAI(
#         base_url=config.get("base_url"),
#         api_key=config.get("api_key"),
#     )
#
#     enhanced_rows: list[dict | None] = [None] * len(indices)
#     failures: list[dict] = []
#     queue = asyncio.Queue()
#     for slot, row_idx in enumerate(indices):
#         queue.put_nowait((slot, row_idx))
#     progress = tqdm(total=len(indices), desc="Enhancing CounterFact")
#
#     async def worker() -> None:
#         while True:
#             try:
#                 slot, row_idx = queue.get_nowait()
#             except asyncio.QueueEmpty:
#                 return
#
#             record = dataset[row_idx]
#             last_errors: list[str] = []
#             last_rejected_output: object | None = None
#             feedback: list[str] | None = None
#             try:
#                 for attempt in range(max_retries + 1):
#                     try:
#                         enhanced, errors, rejected_output = await enhance_record(
#                             record, client, config, feedback
#                         )
#                     except Exception as exc:
#                         enhanced, errors, rejected_output = None, [repr(exc)], None
#
#                     if enhanced is not None:
#                         enhanced_rows[slot] = enhanced
#                         break
#
#                     last_errors = errors
#                     last_rejected_output = rejected_output
#                     feedback = errors
#                     if errors and all(
#                         error.startswith("model returned failed status:")
#                         for error in errors
#                     ):
#                         break
#                     if attempt < max_retries:
#                         await asyncio.sleep(retry_delay * (attempt + 1))
#
#                 if enhanced_rows[slot] is None:
#                     failures.append({
#                         "index": row_idx,
#                         "errors": last_errors,
#                         "record": to_enhancement_record(record),
#                         "enhanced": last_rejected_output,
#                     })
#             finally:
#                 queue.task_done()
#                 progress.update(1)
#
#     tasks = [
#         asyncio.create_task(worker())
#         for _ in range(min(max_concurrency, len(indices)))
#     ]
#     try:
#         await asyncio.gather(*tasks)
#     finally:
#         for task in tasks:
#             if not task.done():
#                 task.cancel()
#         await asyncio.gather(*tasks, return_exceptions=True)
#         progress.close()
#         await client.close()
#
#     kept_rows = [row for row in enhanced_rows if row is not None]
#     enhanced_dataset = (
#         Dataset.from_list(kept_rows, features=dataset.features)
#         if kept_rows else dataset.select([])
#     )
#     failures.sort(key=lambda item: item["index"])
#
#     return enhanced_dataset, failures

In [ ]:
# import os
# from typing import Any
# from dotenv import load_dotenv
#
#
# load_dotenv()
#
# cfact_enhancement_config = {
#     "base_url": "http://localhost:20128/v1",
#     "api_key": os.environ["9ROUTER_API_KEY"],
#     "model": "cx/gpt-5.6-luna(low)",
#     "max_concurrency": 50,
#     "max_retries": 5,
#     "retry_delay": 2.0,
# }
#
# cfact_enhanced, cfact_enhancement_failures = await enhance_dataset(
#     cfact_uniform, cfact_enhancement_config
# )

In [ ]:
# import json
#
#
# cfact_enhanced_path = "../data/datasets/counterfact/enhanced"
# cfact_enhanced.save_to_disk(cfact_enhanced_path)
#
# with open(f"{cfact_enhanced_path}/failures.json", "w", encoding="utf-8") as file:
#     json.dump(cfact_enhancement_failures, file, ensure_ascii=False, indent=2)

In [ ]:
# from datasets import load_from_disk
#
#
# cfact_enhanced = load_from_disk("../data/datasets/counterfact/enhanced")
# cfact_enhanced

### Global Finalization

In [ ]:
# import asyncio
# import json
# import re
# from collections import defaultdict
#
# from datasets import Dataset
# from openai import AsyncOpenAI
# from tqdm.auto import tqdm
#
#
# EDIT_SUBJECT_END = "__edit_subject_end__"
#
#
# def get_words(text: str) -> tuple[str, ...]:
#     return tuple(re.findall(r"\w+", prompt_key(text)))
#
#
# def build_edit_subject_trie(records) -> dict[str, Any]:
#     edit_subject_trie = {}
#     for record_index, record in enumerate(records):
#         edit = record["edit"]
#         node = edit_subject_trie
#         for word in get_words(edit["subject"]):
#             node = node.setdefault(word, {})
#         node.setdefault(EDIT_SUBJECT_END, []).append({
#             "record_index": record_index,
#             "edit_subject": edit["subject"],
#             "edit_subject_key": prompt_key(edit["subject"]),
#             "relation_id": edit["relation_id"],
#         })
#     return edit_subject_trie
#
#
# def find_edit_subject_matches(
#     prompt: str, edit_subject_trie: dict[str, Any]
# ) -> list[dict[str, Any]]:
#     words = get_words(prompt)
#     matches = []
#     for start in range(len(words)):
#         node = edit_subject_trie
#         for end in range(start, len(words)):
#             node = node.get(words[end])
#             if node is None:
#                 break
#             owners = node.get(EDIT_SUBJECT_END, [])
#             if owners:
#                 matches.extend(owners)
#     unique = {}
#     for match in matches:
#         unique[(match["edit_subject_key"], match["relation_id"])] = match
#     return list(unique.values())
#
#
# def contains_edit_subject(
#     prompt: str, relation_id: str, edit_subject_trie: dict[str, Any]
# ) -> bool:
#     return any(
#         match["relation_id"] == relation_id
#         for match in find_edit_subject_matches(prompt, edit_subject_trie)
#     )
#
#
# def build_global_entries(records) -> list[dict[str, Any]]:
#     entries = []
#     for record_index, record in enumerate(records):
#         edit = record["edit"]
#         positive_answer = edit["answer"]["new"]["str"]
#         subject_key = prompt_key(edit["subject"])
#         entries.append({
#             "entry_id": f"{record_index}:edit",
#             "record_index": record_index,
#             "role": "edit",
#             "item_index": None,
#             "subject": edit["subject"],
#             "subject_key": subject_key,
#             "relation_id": edit["relation_id"],
#             "prompt": edit["prompt"],
#             "answer": positive_answer,
#         })
#         for item_index, prompt in enumerate(record["generality"]):
#             entries.append({
#                 "entry_id": f"{record_index}:generality:{item_index}",
#                 "record_index": record_index,
#                 "role": "generality",
#                 "item_index": item_index,
#                 "subject": edit["subject"],
#                 "subject_key": subject_key,
#                 "relation_id": edit["relation_id"],
#                 "prompt": prompt,
#                 "answer": positive_answer,
#             })
#     return entries
#
#
# def build_global_groups(records) -> list[dict[str, Any]]:
#     groups = defaultdict(list)
#     for entry in build_global_entries(records):
#         groups[entry["subject_key"]].append(entry)
#     candidates = []
#     for entries in groups.values():
#         if len({prompt_key(entry["answer"]) for entry in entries}) <= 1:
#             continue
#         candidates.append({
#             "group_id": len(candidates),
#             "subject": entries[0]["subject"],
#             "entries": entries,
#         })
#     return candidates
#
#
# def build_global_audit_prompt(group: dict[str, Any]) -> str:
#     payload = json.dumps(group, ensure_ascii=False, indent=2)
#     return (
#         "Audit CounterFact prompts that mention the same subject.\n"
#         "Find sets of entries that ask the same underlying fact/relation but carry different expected answers.\n"
#         "Do not mark entries that genuinely ask different relations. For example, birthplace and citizenship, or native language and any spoken language, are different relations.\n"
#         "The entries here are edit and generality prompts describing counterfactual positive facts.\n"
#         "Return JSON only with exactly two keys: conflict_sets and reason.\n"
#         "conflict_sets must be a list of lists of entry_id strings. Every set must contain at least two entries with different answers.\n"
#         "Return an empty conflict_sets list when the differing answers belong to genuinely different facts.\n\n"
#         f"GROUP:\n{payload}"
#     )
#
#
# def parse_global_audit_response(text: str, group: dict[str, Any]) -> list[list[str]]:
#     response = text.strip().lstrip("\ufeff")
#     fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", response, re.I | re.S)
#     if fenced:
#         response = fenced.group(1).strip()
#     payload = json.loads(response)
#     if set(payload) != {"conflict_sets", "reason"}:
#         raise ValueError("global audit output keys are invalid")
#     if not isinstance(payload["conflict_sets"], list) or not isinstance(payload["reason"], str):
#         raise ValueError("global audit output types are invalid")
#     entries = {entry["entry_id"]: entry for entry in group["entries"]}
#     conflicts = []
#     for conflict in payload["conflict_sets"]:
#         if not isinstance(conflict, list) or len(conflict) < 2:
#             raise ValueError("each conflict set must contain at least two entry IDs")
#         if any(entry_id not in entries for entry_id in conflict):
#             raise ValueError("global audit returned an unknown entry ID")
#         if len({prompt_key(entries[entry_id]["answer"]) for entry_id in conflict}) < 2:
#             raise ValueError("a conflict set does not contain different answers")
#         conflicts.append(list(dict.fromkeys(conflict)))
#     return conflicts
#
#
# async def audit_global_groups(
#     groups: list[dict[str, Any]],
#     config: dict[str, Any],
# ) -> tuple[list[list[str]], list[dict[str, Any]]]:
#     if not groups or not config.get("enabled", True):
#         return [], []
#     max_concurrency = config.get("max_concurrency", 8)
#     max_retries = config.get("max_retries", 2)
#     retry_delay = config.get("retry_delay", 2.0)
#     if max_concurrency <= 0 or max_retries < 0 or retry_delay < 0:
#         raise ValueError("invalid global audit concurrency/retry configuration")
#
#     client = AsyncOpenAI(
#         base_url=config.get("base_url"),
#         api_key=config.get("api_key"),
#     )
#     queue = asyncio.Queue()
#     for group in groups:
#         queue.put_nowait(group)
#     conflicts = []
#     failures = []
#     progress = tqdm(total=len(groups), desc="Auditing global prompt conflicts")
#
#     async def worker() -> None:
#         while True:
#             try:
#                 group = queue.get_nowait()
#             except asyncio.QueueEmpty:
#                 return
#             last_error = None
#             try:
#                 for attempt in range(max_retries + 1):
#                     try:
#                         completion = await client.chat.completions.create(
#                             model=config["model"],
#                             messages=[
#                                 {
#                                     "role": "system",
#                                     "content": "You audit semantic conflicts in benchmark prompts.",
#                                 },
#                                 {
#                                     "role": "user",
#                                     "content": build_global_audit_prompt(group),
#                                 },
#                             ],
#                             **config.get("request_config", {}),
#                         )
#                         content = completion.choices[0].message.content
#                         conflicts.extend(parse_global_audit_response(content, group))
#                         last_error = None
#                         break
#                     except Exception as exc:
#                         last_error = repr(exc)
#                         if attempt < max_retries:
#                             await asyncio.sleep(retry_delay * (attempt + 1))
#                 if last_error is not None:
#                     failures.append({"group": group, "error": last_error})
#             finally:
#                 queue.task_done()
#                 progress.update(1)
#
#     tasks = [
#         asyncio.create_task(worker())
#         for _ in range(min(max_concurrency, len(groups)))
#     ]
#     try:
#         await asyncio.gather(*tasks)
#     finally:
#         for task in tasks:
#             if not task.done():
#                 task.cancel()
#         await asyncio.gather(*tasks, return_exceptions=True)
#         progress.close()
#         await client.close()
#     failures.sort(key=lambda item: item["group"]["group_id"])
#     return conflicts, failures
#
#
# def finalize_dataset(
#     records,
#     semantic_conflicts: list[list[str]] | tuple[()] = (),
# ) -> tuple[Dataset, dict[str, Any]]:
#     positive_answers = defaultdict(set)
#     positive_owners = defaultdict(set)
#     locality_answers = defaultdict(set)
#     for record_index, record in enumerate(records):
#         answer = record["edit"]["answer"]["new"]["id"]
#         for prompt in [record["edit"]["prompt"], *record["generality"]]:
#             key = prompt_key(prompt)
#             positive_answers[key].add(answer)
#             positive_owners[key].add(record_index)
#         for item in record["locality"]:
#             locality_answers[prompt_key(item["prompt"])].add(
#                 item["answer"]["id"]
#             )
#
#     conflicting_positive = {
#         key for key, answers in positive_answers.items()
#         if len(answers) > 1
#     }
#     conflicting_any = {
#         key
#         for key in set(positive_answers) | set(locality_answers)
#         if len(positive_answers.get(key, set()) | locality_answers.get(key, set())) > 1
#     }
#     exact_drop_records = {
#         record_index
#         for key in conflicting_positive
#         for record_index in positive_owners[key]
#     }
#
#     semantic_drop_records = {
#         int(entry_id.split(":", 1)[0])
#         for conflict in semantic_conflicts
#         for entry_id in conflict
#     }
#
#     edit_subject_trie = build_edit_subject_trie(records)
#     old_prompt_style = re.compile(r",\s*(?:the|a|an)?\s*$", re.I)
#     seen_edits = set()
#     final_rows = []
#     report = {
#         "input_rows": len(records),
#         "conflicting_positive_prompt_keys": len(conflicting_positive),
#         "conflicting_prompt_keys": len(conflicting_any),
#         "exact_conflict_records": len(exact_drop_records),
#         "semantic_conflict_records": len(semantic_drop_records),
#         "duplicate_edit_rows": 0,
#         "old_prompt_style_rows": 0,
#         "removed_generality": 0,
#         "removed_locality": 0,
#         "empty_generality_rows": 0,
#         "empty_locality_rows": 0,
#         "final_rows": 0,
#     }
#
#     for record_index, record in enumerate(records):
#         if record_index in exact_drop_records or record_index in semantic_drop_records:
#             continue
#         edit_key = (
#             prompt_key(record["edit"]["prompt"]),
#             record["edit"]["answer"]["new"]["id"],
#         )
#         if edit_key in seen_edits:
#             report["duplicate_edit_rows"] += 1
#             continue
#         seen_edits.add(edit_key)
#
#         all_prompts = [
#             record["edit"]["prompt"],
#             *record["generality"],
#             *(item["prompt"] for item in record["locality"]),
#         ]
#         if any(old_prompt_style.search(prompt) for prompt in all_prompts):
#             report["old_prompt_style_rows"] += 1
#             continue
#
#         generality = list(dict.fromkeys(record["generality"]))
#         report["removed_generality"] += len(record["generality"]) - len(generality)
#         if not generality:
#             report["empty_generality_rows"] += 1
#             continue
#
#         positive_keys = {
#             prompt_key(record["edit"]["prompt"]),
#             *(prompt_key(prompt) for prompt in generality),
#         }
#         locality = []
#         seen_locality = set()
#         for item_index, item in enumerate(record["locality"]):
#             key = prompt_key(item["prompt"])
#             if (
#                 key in conflicting_any
#                 or key in positive_keys
#                 or any(owner != record_index for owner in positive_owners.get(key, set()))
#                 or contains_edit_subject(
#                     item["prompt"],
#                     record["edit"]["relation_id"],
#                     edit_subject_trie,
#                 )
#                 or key in seen_locality
#             ):
#                 report["removed_locality"] += 1
#                 continue
#             seen_locality.add(key)
#             locality.append({
#                 "prompt": item["prompt"],
#                 "answer": item["answer"],
#             })
#         if not locality:
#             report["empty_locality_rows"] += 1
#             continue
#
#         final_rows.append({
#             "edit": record["edit"],
#             "generality": generality,
#             "locality": locality,
#         })
#
#     report["final_rows"] = len(final_rows)
#     return Dataset.from_list(final_rows), report

In [ ]:
# import json
# import os
# from dotenv import load_dotenv
#
#
# load_dotenv()
#
# cfact_global_audit_config = {
#     "enabled": True,
#     "base_url": "http://localhost:20128/v1",
#     "api_key": os.environ["9ROUTER_API_KEY"],
#     "model": "cx/gpt-5.6-luna(low)",
#     "max_concurrency": 50,
#     "max_retries": 2,
#     "retry_delay": 2.0,
# }
#
#
# cfact_global_groups = build_global_groups(cfact_enhanced)
# cfact_semantic_conflicts, cfact_global_audit_failures = await audit_global_groups(
#     cfact_global_groups, cfact_global_audit_config
# )
# if cfact_global_audit_failures:
#     raise RuntimeError(
#         f"Global audit failed for {len(cfact_global_audit_failures)} groups; "
#         "inspect cfact_global_audit_failures before finalizing"
#     )
# cfact_final, cfact_final_report = finalize_dataset(
#     cfact_enhanced, cfact_semantic_conflicts
# )
# cfact_final_report["global_groups"] = len(cfact_global_groups)
# cfact_final_report["semantic_conflict_sets"] = len(cfact_semantic_conflicts)
# cfact_final_report["global_audit_failures"] = len(cfact_global_audit_failures)
#
# cfact_final_path = "../data/datasets/counterfact/final"
# cfact_final.save_to_disk(cfact_final_path)
# with open(f"{cfact_final_path}/report.json", "w", encoding="utf-8") as file:
#     json.dump(cfact_final_report, file, ensure_ascii=False, indent=2)
# with open(f"{cfact_final_path}/global_audit_failures.json", "w", encoding="utf-8") as file:
#     json.dump(cfact_global_audit_failures, file, ensure_ascii=False, indent=2)
# cfact_final_report

In [ ]:
# from datasets import load_from_disk
#
#
# cfact = load_from_disk("../data/datasets/counterfact/final")
# cfact

## MQuAKE-Remastered

### Raw Snapshot

In [ ]:
# from datasets import load_dataset


# mquake_raw = load_dataset("henryzhongsc/MQuAKE-Remastered")
# mquake_raw.save_to_disk("../data/datasets/mquake-remastered/raw")

In [ ]:
# from datasets import load_from_disk


# mquake_raw = load_from_disk("../data/datasets/mquake-remastered/raw")
# mquake_raw

### Merge And Deduplication

In [ ]:
# from datasets import Dataset, concatenate_datasets
# from typing import Any


# def get_key(record: dict[str, Any]) -> tuple[tuple[str, str, str], ...]:
#     rewrites = record["requested_rewrite"]

#     return tuple(sorted((
#         rewrite["subject"],
#         rewrite["relation_id"],
#         rewrite["target_new_id"]
#     ) for rewrite in rewrites))


# def get_score(record: dict[str, Any]) -> tuple[int, int]:
#     return (
#         len(record["requested_rewrite"]),
#         len(record["single_hops"])
#     )


# mquake_merged = concatenate_datasets([mquake_raw["CF9k"], mquake_raw["CF3k"], mquake_raw["CF6334"], mquake_raw["T"]])

# true_target_ids_by_key = {}
# for record in mquake_merged:
#     key = get_key(record)
#     true_target_ids = true_target_ids_by_key.get(key, set())
#     true_target_ids.add(tuple(sorted(rr["target_true_id"] for rr in record["requested_rewrite"])))
#     true_target_ids_by_key[key] = true_target_ids

# mquake_merged = mquake_merged.filter(lambda r: len(true_target_ids_by_key[get_key(r)]) == 1)

# records_by_key = {}
# for record in mquake_merged:
#     key = get_key(record)
#     score = get_score(record)
#     records = records_by_key.get(key, []) + [{"score": score, "record": record}]
#     records_by_key[key] = records

# records_by_key = {k: sorted(v, key=lambda r: r["score"]) for k,v in records_by_key.items()}

# mquake_merged = Dataset.from_list([v[0]["record"] for v in records_by_key.values()])
# mquake_merged.save_to_disk("../data/datasets/mquake-remastered/merged")

In [ ]:
# from datasets import load_from_disk


# mquake_merged = load_from_disk("../data/datasets/mquake-remastered/merged")
# mquake_merged

### Evaluation Formatting

In [ ]:
# def norm(text) -> str:
#     return " ".join(str(text or "").casefold().strip().split())


# def matched(record: dict[str, Any], edit_idx: int, hop_idx: int) -> bool:
#     rewrite = record["requested_rewrite"][edit_idx]

#     edit_id = tuple(record["edit_triples"][edit_idx])
#     hop_id = tuple(record["new_triples"][hop_idx])

#     if edit_id == hop_id:
#         return True

#     edit_str = (
#         norm(rewrite["subject"]),
#         rewrite["relation_id"],
#         norm(rewrite["target_new_str"]),
#     )
#     hop_str = (
#         norm(record["new_triples_labeled"][hop_idx][0]),
#         record["new_triples"][hop_idx][1],
#         norm(record["new_triples_labeled"][hop_idx][2]),
#     )

#     return edit_str == hop_str


# def get_edit_pos(record: dict[str, Any], edit_idx: int = 0) -> int:
#     positions = [
#         hop_idx
#         for hop_idx in range(len(record["new_triples"]))
#         if matched(record, edit_idx, hop_idx)
#     ]

#     if len(positions) != 1:
#         raise ValueError(f"Expected exactly 1 edit position, got {positions}")

#     return positions[0]


# def get_edit(record: dict[str, Any], has_single_edit: bool) -> list[tuple[str, str]]:
#     edit_pos = get_edit_pos(record) if has_single_edit else 0
#     edit_ids = [
#         *record["orig_triples"][edit_pos],
#         record["new_triples"][edit_pos][2]
#     ]
#     edit_labels = [
#         *record["orig_triples_labeled"][edit_pos],
#         record["new_triples_labeled"][edit_pos][2]
#     ]
#     return [(id, label) for id, label in zip(edit_ids, edit_labels)]


# def get_hop_triples(
#     record: dict[str, Any],
#     type: str,
#     has_single_edit: bool,
# ) -> list[list[tuple[str, str]]]:
#     hop_triples = [
#         [(id, label) for id, label in zip(ids, labels)]
#         for ids, labels in zip(record[f"{type}_triples"], record[f"{type}_triples_labeled"])
#     ]
#     if type == "new" and not has_single_edit:
#         rewrites = record["requested_rewrite"]
#         hop_triples[1][2] = (rewrites[1]["target_true_id"],
#                              rewrites[1]["target_true_str"])
#     return hop_triples


# def bundle_answers(obj: dict[str, Any], key: str) -> list[str]:
#     values = [obj[key], *obj[f"{key}_alias"]]
#     ans, seen = [], set()

#     for value in values:
#         value = value.strip()
#         marker = norm(value)
#         if value and marker not in seen:
#             seen.add(marker)
#             ans.append(value)

#     return ans


# def uniform(record: dict[str, Any]) -> dict[str, Any]:
#     rewrites = record["requested_rewrite"]
#     has_single_edit = len(rewrites) == 1

#     edit = get_edit(record, has_single_edit)
#     true_hop_triples = get_hop_triples(record, "orig", has_single_edit)
#     new_hop_triples = get_hop_triples(record, "new", has_single_edit)

#     return {
#         "edit": {
#             "prompt": rewrites[0]["prompt"].format(rewrites[0]["subject"]),
#             "subject": {
#                 "id": edit[0][0],
#                 "str": edit[0][1]
#             },
#             "relation": {
#                 "id": edit[1][0],
#                 "str": edit[1][1]
#             },
#             "answer": {
#                 "true": {
#                     "id": edit[2][0],
#                     "str": edit[2][1]
#                 },
#                 "new": {
#                     "id": edit[3][0],
#                     "str": edit[3][1]
#                 }
#             }
#         },
#         "portability": {
#             "rewrite_required": len(record['single_hops']) > 2,
#             "prompts": record["questions"],
#             "answers": {
#                 "true": bundle_answers(record["single_hops"][1], "answer"),
#                 "new": bundle_answers(record["new_single_hops"][1], "answer")
#                        if has_single_edit else
#                        [rewrites[1]["target_true_str"]]
#             },
#             "hops": {
#                 "true": [{
#                     "prompt": hop["cloze"],
#                     "subject": {
#                         "id": true_hop_triples[idx][0][0],
#                         "str": true_hop_triples[idx][0][1]
#                     },
#                     "relation": {
#                         "id": true_hop_triples[idx][1][0],
#                         "str": true_hop_triples[idx][1][1]},
#                     "answers": {
#                         "id": true_hop_triples[idx][2][0],
#                         "str": bundle_answers(hop, "answer")
#                     }
#                 } for idx, hop in enumerate(record["single_hops"][:2])],
#                 "new": [{
#                     "prompt": hop["cloze"],
#                     "subject": {
#                         "id": new_hop_triples[idx][0][0],
#                         "str": new_hop_triples[idx][0][1]
#                     },
#                     "relation": {
#                         "id": new_hop_triples[idx][1][0],
#                         "str": new_hop_triples[idx][1][1]},
#                     "answers": {
#                         "id": new_hop_triples[idx][2][0],
#                         "str": bundle_answers(hop, "answer")
#                                if idx != 1 or has_single_edit
#                                else [rewrites[1]["target_true_str"]]
#                     }
#                 } for idx, hop in enumerate(record["new_single_hops"][:2])]
#             }
#         }
#     }


# mquake_uniform = mquake_merged\
#     .filter(lambda r:
#         len(set(r["questions"])) == 3 and
#         ((len(r["requested_rewrite"]) == 1 and get_edit_pos(r) < 2) or
#          len(r["requested_rewrite"]) == len(r["single_hops"]))
#     )\
#     .map(uniform, remove_columns=mquake_merged.column_names)

# mquake_uniform.save_to_disk("../data/datasets/mquake-remastered/uniform")

In [ ]:
# from datasets import load_from_disk


# mquake_uniform = load_from_disk("../data/datasets/mquake-remastered/uniform")
# mquake_uniform

### Answer Cleaning

In [ ]:
# import re


# def norm(text) -> str:
#     text = str(text or "").casefold()
#     text = re.sub(r"[^0-9a-z]+", " ", text)
#     return " ".join(text.split())


# def has_port_ans_overlapped(record: dict[str, Any]) -> set[str]:
#     port_ans = record["portability"]["answers"]
#     true_port_ans = set(v for v in map(norm, port_ans["true"]) if v)
#     new_port_ans = set(v for v in map(norm, port_ans["new"]) if v)
#     return true_port_ans & new_port_ans


# mquake_clean = mquake_uniform.filter(lambda r: not has_port_ans_overlapped(r))
# mquake_clean.save_to_disk("../data/datasets/mquake-remastered/clean")

In [ ]:
# from datasets import load_from_disk


# mquake_clean = load_from_disk("../data/datasets/mquake-remastered/clean")
# mquake_clean

### Prompt Rewriting

In [ ]:
# import asyncio
# import json
# import re
# from copy import deepcopy
# from typing import Any

# from datasets import Dataset
# from openai import AsyncOpenAI
# from tqdm.auto import tqdm


# EXPECTED_MQUAKE_PORTABILITY = 3


# def normalize_prompt(text) -> str:
#     return " ".join(str(text or "").strip().split())


# def prompt_key(text) -> str:
#     text = re.sub(r"[^0-9a-z]+", " ", str(text or "").casefold())
#     return " ".join(text.split())


# def contains_phrase(prompt: str, phrase: str, min_len: int = 1) -> bool:
#     phrase_key = prompt_key(phrase)
#     if len(phrase_key) < min_len:
#         return False
#     return f" {phrase_key} " in f" {prompt_key(prompt)} "


# def answer_leaks_into_prompt(prompt: str, answers: list[str]) -> str | None:
#     for answer in answers:
#         if contains_phrase(prompt, answer, min_len=4):
#             return answer
#     return None


# def strip_json_response(text: str) -> str:
#     text = str(text or "").strip()
#     if text.startswith("```"):
#         text = re.sub(r"^```(?:json)?\s*", "", text)
#         text = re.sub(r"\s*```$", "", text)
#     return text.strip()


# def build_canonical_prompt(record: dict[str, Any]) -> str:
#     hop1, hop2 = record["portability"]["hops"]["new"][:2]
#     start_subject = hop1["subject"]["str"]
#     hop1_relation = hop1["relation"]["str"]
#     hop2_relation = hop2["relation"]["str"]
#     return f"What is the {hop2_relation} of the {hop1_relation} of {start_subject}?"


# def compact_mquake_record(record: dict[str, Any], include_prompts: bool = True) -> dict[str, Any]:
#     portability = record["portability"]
#     compact = {
#         "canonical_prompt": build_canonical_prompt(record),
#         "expected_answers": list(portability["answers"]["new"]),
#         "hops": [
#             {
#                 "subject": hop["subject"]["str"],
#                 "relation": hop["relation"]["str"],
#                 "answer": hop["answers"]["str"][0],
#             }
#             for hop in portability["hops"]["new"]
#         ],
#     }
#     if include_prompts:
#         compact["prompts"] = list(portability.get("prompts", []))
#     return compact


# def finalize_mquake_record(record: dict[str, Any]) -> dict[str, Any]:
#     record = deepcopy(dict(record))
#     record["portability"] = deepcopy(dict(record["portability"]))
#     record["portability"].pop("rewrite_required", None)
#     return record


# def validate_rewritten_prompts(record: dict[str, Any], prompts: list[str]) -> list[str]:
#     errors: list[str] = []
#     prompts = [normalize_prompt(prompt) for prompt in prompts]

#     if len(prompts) != EXPECTED_MQUAKE_PORTABILITY:
#         return [f"expected {EXPECTED_MQUAKE_PORTABILITY} prompts, got {len(prompts)}"]

#     keys = [prompt_key(prompt) for prompt in prompts]
#     if len(set(keys)) != len(keys):
#         errors.append("duplicate prompts")

#     expected_answers = record["portability"]["answers"]["new"]
#     chain_start = record["portability"]["hops"]["new"][0]["subject"]["str"]
#     chain_start_key = prompt_key(chain_start)

#     for idx, prompt in enumerate(prompts):
#         lowered = prompt.casefold()
#         if not prompt:
#             errors.append(f"prompt {idx}: empty")
#             continue
#         if not prompt.endswith("?"):
#             errors.append(f"prompt {idx}: not a question")
#         if re.match(r"^\s*(\d+[).:-]|[-*])\s+", prompt):
#             errors.append(f"prompt {idx}: contains bullet or numbering")
#         if any(token in lowered for token in ("```", "prompt:", "answer:", "record ", "question 1")):
#             errors.append(f"prompt {idx}: contains labels or markdown")
#         if chain_start_key and chain_start_key not in prompt_key(prompt):
#             errors.append(f"prompt {idx}: missing chain-start subject `{chain_start}`")

#         leaked_answer = answer_leaks_into_prompt(prompt, expected_answers)
#         if leaked_answer:
#             errors.append(f"prompt {idx}: leaks expected answer `{leaked_answer}`")
#     return errors


# def select_mquake_examples(dataset, max_examples: int = 4) -> list[dict[str, Any]]:
#     examples = []
#     for record in dataset:
#         if record["portability"].get("rewrite_required"):
#             continue
#         errors = validate_rewritten_prompts(record, record["portability"].get("prompts", []))
#         if errors:
#             continue
#         examples.append(compact_mquake_record(record, include_prompts=True))
#         if len(examples) >= max_examples:
#             break
#     return examples


# def build_rewrite_prompt(record: dict[str, Any], examples: list[dict[str, Any]]) -> str:
#     example_payload = json.dumps(examples, ensure_ascii=False, indent=2)
#     task_payload = json.dumps(compact_mquake_record(record, include_prompts=False), ensure_ascii=False, indent=2)
#     return (
#         "Paraphrase one canonical MQuAKE portability question for a model-editing benchmark.\n"
#         "Only write prompt strings. Hop subjects, relations, answers, and list lengths are locked.\n"
#         "The input has a two-hop chain and a canonical_prompt. The canonical_prompt is the source of truth.\n"
#         "The target record's original prompts are intentionally omitted to avoid copying obsolete longer-chain wording.\n"
#         "Your 3 questions must ask exactly the same thing as canonical_prompt: the direct answer of hop 2 after following hop 1.\n"
#         "Return one JSON object containing only a prompts list.\n\n"
#         "Hard semantic rule:\n"
#         "The expected answer is the direct answer to hop 2. Do not ask for any property of that answer.\n"
#         "For example, if hop 2 is `cricket -- country of origin --> England`, ask for the country of origin, not England's continent, capital, language, head of state, or head of government.\n"
#         "If hop 2 is `Helsinki -- official language --> Esperanto`, ask for the official language, not who created Esperanto or where Esperanto is spoken.\n\n"
#         "Rules:\n"
#         "1) Return exactly 3 natural questions.\n"
#         "2) Each question must preserve the exact target meaning of canonical_prompt.\n"
#         "3) Each question must require the two-hop chain from the chain-start subject through hop 1, then hop 2.\n"
#         "4) Include the chain-start subject exactly as written.\n"
#         "5) Do not directly name the intermediate entity unless unavoidable; prefer phrases such as 'the sport associated with X' or 'the author of X'.\n"
#         "6) Do not include, paraphrase, hint, or leak any expected answer.\n"
#         "7) Keep all prompts as questions ending with a question mark.\n"
#         "8) Do not add labels, numbering, bullets, explanations, markdown, or extra text inside prompts.\n"
#         "9) The 3 questions must not be duplicates or near-identical copies.\n\n"
#         "Valid examples that already satisfy the target style. Use them only for style, never for content:\n"
#         f"{example_payload}\n\n"
#         "Input record to paraphrase:\n"
#         f"{task_payload}\n\n"
#         "Output JSON only. Format:\n"
#         "{\"prompts\": [\"question 1?\", \"question 2?\", \"question 3?\"]}"
#     )


# def parse_rewrite_response(content: str) -> list[str]:
#     content = strip_json_response(content)
#     try:
#         payload = json.loads(content)
#     except Exception:
#         start = content.find("{")
#         end = content.rfind("}")
#         if start == -1 or end == -1 or end < start:
#             raise ValueError("response does not contain a JSON object")
#         payload = json.loads(content[start:end + 1])

#     if not isinstance(payload, dict) or not isinstance(payload.get("prompts"), list):
#         raise ValueError("response must be a JSON object with a prompts list")

#     return [normalize_prompt(prompt) for prompt in payload["prompts"]]


# async def rewrite_mquake_record(
#     record: dict[str, Any],
#     examples: list[dict[str, Any]],
#     client: AsyncOpenAI,
#     config: dict[str, Any],
# ) -> tuple[list[str] | None, list[str], dict[str, Any] | None]:
#     messages = [
#         {
#             "role": "system",
#             "content": "You paraphrase benchmark questions. You only change prompt text and return valid JSON.",
#         },
#         {
#             "role": "user",
#             "content": build_rewrite_prompt(record, examples),
#         },
#     ]

#     completion = await client.chat.completions.create(
#         model=config["model"],
#         messages=messages,
#         **config.get("request_config", {}),
#     )

#     content = completion.choices[0].message.content
#     if content is None:
#         return None, ["empty model response"], None

#     try:
#         prompts = parse_rewrite_response(content)
#     except Exception as exc:
#         return None, [f"invalid JSON: {exc}"], {"raw_response": strip_json_response(content)}

#     errors = validate_rewritten_prompts(record, prompts)
#     if errors:
#         return None, errors, {"prompts": prompts}

#     return prompts, [], None


# async def rewrite_mquake_dataset(
#     dataset,
#     config: dict[str, Any],
#     start: int = 0,
#     limit: int | None = None,
# ) -> tuple[Dataset, list[dict[str, Any]]]:
#     if not config.get("model"):
#         raise ValueError("config['model'] must be specified")
#     if start < 0:
#         raise ValueError("start must be non-negative")
#     if limit is not None and limit < 0:
#         raise ValueError("limit must be non-negative when set")

#     end = len(dataset) if limit is None else min(len(dataset), start + limit)
#     source_indices = list(range(start, end))
#     rows = [deepcopy(dict(dataset[idx])) for idx in source_indices]
#     examples = select_mquake_examples(dataset, max_examples=4)
#     tasks = [
#         (slot, source_idx, record)
#         for slot, (source_idx, record) in enumerate(zip(source_indices, rows))
#         if record["portability"].get("rewrite_required")
#     ]

#     client = AsyncOpenAI(
#         base_url=config.get("base_url"),
#         api_key=config.get("api_key"),
#     )
#     semaphore = asyncio.Semaphore(config.get("max_concurrency", 4))
#     max_retries = config.get("max_retries", 3)
#     retry_delay = config.get("retry_delay", 2.0)

#     failures: list[dict] = []
#     rewritten_indices: set[int] = set()

#     async def worker(slot: int, source_idx: int, record: dict[str, Any]) -> None:
#         last_errors: list[str] = []
#         last_rejected_output: object | None = None

#         async with semaphore:
#             for attempt in range(max_retries + 1):
#                 try:
#                     prompts, errors, rejected_output = await rewrite_mquake_record(record, examples, client, config)
#                 except Exception as exc:
#                     prompts, errors, rejected_output = None, [repr(exc)], None

#                 if prompts is not None:
#                     rows[slot]["portability"]["prompts"] = prompts
#                     rewritten_indices.add(source_idx)
#                     return

#                 last_errors = errors
#                 last_rejected_output = rejected_output
#                 if attempt < max_retries:
#                     await asyncio.sleep(retry_delay * (attempt + 1))

#         failures.append({
#             "index": source_idx,
#             "errors": last_errors,
#             "record": compact_mquake_record(record, include_prompts=False),
#             "rewritten": last_rejected_output,
#         })

#     rewrite_tasks = [worker(slot, source_idx, record) for slot, source_idx, record in tasks]
#     for task in tqdm(asyncio.as_completed(rewrite_tasks), total=len(rewrite_tasks), desc="Rewriting MQuAKE"):
#         await task

#     kept_rows = []
#     final_failures = list(failures)
#     failed_indices = {failure["index"] for failure in failures}

#     for source_idx, row in zip(source_indices, rows):
#         if source_idx in failed_indices:
#             continue

#         errors = validate_rewritten_prompts(row, row["portability"].get("prompts", []))
#         if errors:
#             final_failures.append({
#                 "index": source_idx,
#                 "errors": errors,
#                 "record": compact_mquake_record(row, include_prompts=False),
#                 "rewritten": {"prompts": row["portability"].get("prompts", [])},
#             })
#             continue
#         kept_rows.append(finalize_mquake_record(row))

#     report = {
#         "dataset_records": len(dataset),
#         "selected_start": start,
#         "selected_end": end,
#         "selected_records": len(rows),
#         "original_records": len(rows),
#         "rewrite_required_records": len(tasks),
#         "rewritten_records": len(rewritten_indices),
#         "kept_records": len(kept_rows),
#         "failed_records": len(final_failures),
#     }
#     return Dataset.from_list(kept_rows), final_failures, report

In [ ]:
# import os
# from dotenv import load_dotenv


# load_dotenv("../.env")

# llm_rewrite_config = {
#     "base_url": "https://openrouter.ai/api/v1",
#     "api_key": os.environ["9ROUTER_API_KEY"],
#     "model": "deepseek/deepseek-v4-flash",
#     "request_config": {
#         "temperature": 1.0,
#         "top_p": 1.0,
#         "extra_body": {
#             "thinking": {"type": "disabled"},
#         },
#     },
#     "max_concurrency": 50,
#     "max_retries": 5,
#     "retry_delay": 2.0,
# }

# mquake_rewritten, mquake_rewrite_failures, mquake_rewrite_report = await rewrite_mquake_dataset(
#     mquake_clean,
#     llm_rewrite_config,
#     start=0,
#     limit=None
# )
# mquake_rewrite_report

In [ ]:
# import json


# mquake_rewritten.save_to_disk("../data/datasets/mquake-remastered/rewritten")

# with open("../data/datasets/mquake-remastered/rewritten/failures.json", "w", encoding="utf-8") as file:
#     json.dump(mquake_rewrite_failures, file, ensure_ascii=False, indent=2)

In [ ]:
# from datasets import load_from_disk


# mquake_rewritten = load_from_disk("../data/datasets/mquake-remastered/rewritten")
# mquake_rewritten

### Final Validation

In [ ]:
# import json
# import re
# import shutil
# from copy import deepcopy
# from pathlib import Path

# from datasets import Dataset


# EXPECTED_MQUAKE_PORTABILITY = 3
# MQUAKE_FINAL_PATH = Path("../data/datasets/mquake-remastered/final")
# MQUAKE_INSPECTION_PATH = Path("../data/datasets/mquake-remastered/inspection")
# BAD_P136_FIRST_HOP_ROLES = {
#     "P50": "author",
#     "P170": "creator",
#     "P112": "founder",
# }


# def normalize_prompt(text) -> str:
#     return " ".join(str(text or "").strip().split())


# def prompt_key(text) -> str:
#     text = re.sub(r"[^0-9a-z]+", " ", str(text or "").casefold())
#     return " ".join(text.split())


# def contains_phrase(prompt: str, phrase: str, min_len: int = 1) -> bool:
#     phrase_key = prompt_key(phrase)
#     if len(phrase_key) < min_len:
#         return False
#     return f" {phrase_key} " in f" {prompt_key(prompt)} "


# def answer_leaks_into_prompt(prompt: str, answers: list[str]) -> str | None:
#     for answer in answers:
#         if contains_phrase(prompt, answer, min_len=4):
#             return answer
#     return None


# def save_dataset_replace(dataset, path: str) -> None:
#     path = Path(path)
#     tmp_path = path.with_name(f"{path.name}_tmp")
#     if tmp_path.exists():
#         shutil.rmtree(tmp_path)
#     dataset.save_to_disk(tmp_path)
#     if path.exists():
#         shutil.rmtree(path)
#     tmp_path.rename(path)


# def has_bad_p136_music_construction(record: dict[str, Any]) -> bool:
#     portability = record["portability"]
#     first_hop_relation = portability["hops"]["true"][0]["relation"]["id"]
#     last_hop_relations = {
#         portability["hops"]["true"][-1]["relation"]["id"],
#         portability["hops"]["new"][-1]["relation"]["id"],
#     }
#     prompt_text = " ".join(portability.get("prompts", [])).casefold()
#     return (
#         first_hop_relation in BAD_P136_FIRST_HOP_ROLES
#         and "P136" in last_hop_relations
#         and "music" in prompt_text
#         and ("play" in prompt_text or "played" in prompt_text)
#     )


# def rewrite_bad_p136_prompts(record: dict[str, Any]) -> tuple[dict[str, Any], bool]:
#     if not has_bad_p136_music_construction(record):
#         return deepcopy(dict(record)), False

#     record = deepcopy(dict(record))
#     record["portability"] = deepcopy(dict(record["portability"]))
#     first_hop = record["portability"]["hops"]["true"][0]
#     role = BAD_P136_FIRST_HOP_ROLES[first_hop["relation"]["id"]]
#     subject = first_hop["subject"]["str"]
#     record["portability"]["prompts"] = [
#         f"What genre is the {role} of {subject} known for?",
#         f"Which creative genre is most associated with the {role} of {subject}?",
#         f"In what genre does the {role} of {subject} mainly work?",
#     ]
#     return record, True


# def finalize_mquake_record(record: dict[str, Any]) -> dict[str, Any]:
#     record = deepcopy(dict(record))
#     record["portability"] = deepcopy(dict(record["portability"]))
#     record["portability"].pop("rewrite_required", None)
#     return record


# def validate_mquake_final_record(record: dict[str, Any]) -> list[str]:
#     errors = []
#     prompts = [normalize_prompt(prompt) for prompt in record["portability"].get("prompts", [])]

#     if len(prompts) != EXPECTED_MQUAKE_PORTABILITY:
#         return [f"expected {EXPECTED_MQUAKE_PORTABILITY} prompts, got {len(prompts)}"]

#     keys = [prompt_key(prompt) for prompt in prompts]
#     if len(set(keys)) != len(keys):
#         errors.append("duplicate prompts")

#     expected_answers = record["portability"]["answers"]["new"]
#     chain_start = record["portability"]["hops"]["new"][0]["subject"]["str"]
#     chain_start_key = prompt_key(chain_start)

#     for idx, prompt in enumerate(prompts):
#         lowered = prompt.casefold()
#         if not prompt:
#             errors.append(f"prompt {idx}: empty")
#             continue
#         if not prompt.endswith("?"):
#             errors.append(f"prompt {idx}: not a question")
#         if re.match(r"^\s*(\d+[).:-]|[-*])\s+", prompt):
#             errors.append(f"prompt {idx}: contains bullet or numbering")
#         if any(token in lowered for token in ("```", "prompt:", "answer:", "record ", "question 1")):
#             errors.append(f"prompt {idx}: contains labels or markdown")
#         if chain_start_key and chain_start_key not in prompt_key(prompt):
#             errors.append(f"prompt {idx}: missing chain-start subject `{chain_start}`")

#         leaked_answer = answer_leaks_into_prompt(prompt, expected_answers)
#         if leaked_answer:
#             errors.append(f"prompt {idx}: leaks expected answer `{leaked_answer}`")

#     if has_bad_p136_music_construction(record):
#         errors.append("bad P136 music/play construction remains")

#     return errors


# def get_mquake_edit_pair(record: dict[str, Any]) -> tuple[str, str]:
#     edit = record["edit"]
#     return edit["subject"]["id"], edit["relation"]["id"]


# def get_mquake_edit_label_pair(record: dict[str, Any]) -> tuple[str, str]:
#     edit = record["edit"]
#     return prompt_key(edit["subject"]["str"]), edit["relation"]["id"]


# def mquake_hop_matches_edit(edit: dict[str, Any], hop: dict[str, Any]) -> bool:
#     exact_match = (
#         hop["subject"]["id"] == edit["subject"]["id"]
#         and hop["relation"]["id"] == edit["relation"]["id"]
#         and hop["answers"]["id"] == edit["answer"]["new"]["id"]
#     )
#     lexical_match = (
#         prompt_key(hop["subject"]["str"]) == prompt_key(edit["subject"]["str"])
#         and hop["relation"]["id"] == edit["relation"]["id"]
#         and prompt_key(edit["answer"]["new"]["str"])
#             in {prompt_key(value) for value in hop["answers"]["str"]}
#     )
#     return exact_match or lexical_match


# def get_mquake_unrelated_hop(record: dict[str, Any]) -> dict[str, Any]:
#     edit = record["edit"]
#     unrelated = [
#         hop for hop in record["portability"]["hops"]["new"]
#         if not mquake_hop_matches_edit(edit, hop)
#     ]
#     if len(unrelated) != 1:
#         raise ValueError(f"Expected exactly one unrelated hop, got {len(unrelated)}")
#     return unrelated[0]


# mquake_edit_pair_indices = {}
# mquake_edit_label_pair_indices = {}
# for idx, record in enumerate(mquake_rewritten):
#     mquake_edit_pair_indices.setdefault(get_mquake_edit_pair(record), set()).add(idx)
#     mquake_edit_label_pair_indices.setdefault(
#         get_mquake_edit_label_pair(record), set()
#     ).add(idx)

# mquake_final_rows = []
# mquake_final_failures = []
# mquake_p136_fixed = []
# mquake_cross_edit_skipped = 0
# mquake_duplicate_edit_skipped = 0
# mquake_selected_edit_pairs = set()

# for idx, record in enumerate(mquake_rewritten):
#     fixed_record, was_fixed = rewrite_bad_p136_prompts(record)
#     fixed_record = finalize_mquake_record(fixed_record)
#     errors = validate_mquake_final_record(fixed_record)
#     if errors:
#         mquake_final_failures.append({
#             "index": idx,
#             "errors": errors,
#             "record": fixed_record,
#         })
#         continue

#     try:
#         unrelated_hop = get_mquake_unrelated_hop(fixed_record)
#     except ValueError as exc:
#         mquake_final_failures.append({
#             "index": idx,
#             "errors": [str(exc)],
#             "record": fixed_record,
#         })
#         continue
#     unrelated_pair = (
#         unrelated_hop["subject"]["id"],
#         unrelated_hop["relation"]["id"],
#     )
#     unrelated_label_pair = (
#         prompt_key(unrelated_hop["subject"]["str"]),
#         unrelated_hop["relation"]["id"],
#     )
#     if (
#         unrelated_pair in mquake_edit_pair_indices
#         or unrelated_label_pair in mquake_edit_label_pair_indices
#     ):
#         mquake_cross_edit_skipped += 1
#         continue
#     edit_pair = get_mquake_edit_pair(fixed_record)
#     if edit_pair in mquake_selected_edit_pairs:
#         mquake_duplicate_edit_skipped += 1
#         continue
#     mquake_selected_edit_pairs.add(edit_pair)

#     if was_fixed:
#         mquake_p136_fixed.append({
#             "index": idx,
#             "before": list(record["portability"].get("prompts", [])),
#             "after": list(fixed_record["portability"].get("prompts", [])),
#             "answers": dict(fixed_record["portability"]["answers"]),
#             "hops": dict(fixed_record["portability"]["hops"]),
#         })
#     mquake_final_rows.append(fixed_record)

# mquake_final = Dataset.from_list(mquake_final_rows)
# save_dataset_replace(mquake_final, MQUAKE_FINAL_PATH)

# MQUAKE_INSPECTION_PATH.mkdir(parents=True, exist_ok=True)
# with open(MQUAKE_INSPECTION_PATH / "mquake_p136_fixed_examples.json", "w", encoding="utf-8") as file:
#     json.dump(mquake_p136_fixed[:25], file, ensure_ascii=False, indent=2)

# with open(MQUAKE_INSPECTION_PATH / "mquake_final_failures.json", "w", encoding="utf-8") as file:
#     json.dump(mquake_final_failures, file, ensure_ascii=False, indent=2)

# mquake_final_report = {
#     "rewritten_rows": len(mquake_rewritten),
#     "p136_fixed_rows": len(mquake_p136_fixed),
#     "cross_edit_rows": mquake_cross_edit_skipped,
#     "duplicate_edit_rows": mquake_duplicate_edit_skipped,
#     "final_rows": len(mquake_final),
#     "failed_rows": len(mquake_final_failures),
#     "final_path": str(MQUAKE_FINAL_PATH),
# }

# with open(MQUAKE_INSPECTION_PATH / "mquake_final_report.json", "w", encoding="utf-8") as file:
#     json.dump(mquake_final_report, file, ensure_ascii=False, indent=2)

# mquake_final_report

In [ ]:
# from datasets import load_from_disk


# mquake = load_from_disk("../data/datasets/mquake-remastered/final")
# mquake

## UniEdit

### Raw Snapshot

In [ ]:
# from huggingface_hub import snapshot_download


# snapshot_download(
#     repo_id="qizhou/UniEdit",
#     repo_type="dataset",
#     local_dir="../data/datasets/uniedit/raw",
#     allow_patterns=["*.json"],
# )

In [ ]:
# from glob import glob


# uniedit_raw = {
#     split: sorted(glob(f"../data/datasets/uniedit/raw/{split}/*.json"))
#     for split in ("train", "test")
# }
# {split: len(source_files) for split, source_files in uniedit_raw.items()}

### Merge Repeated Edits

In [ ]:
# import json
# import os
# from tqdm.auto import tqdm
# from typing import Any


# def normalize_uniedit_text(text) -> str:
#     return " ".join(str(text).split()).casefold()


# def extend_unique_uniedit_texts(output: list[str], values: list[Any]) -> None:
#     seen = {normalize_uniedit_text(value) for value in output}
#     for value in values:
#         if value is None or not str(value).strip():
#             continue
#         value = str(value).strip()
#         key = normalize_uniedit_text(value)
#         if key not in seen:
#             seen.add(key)
#             output.append(value)


# def get_uniedit_tail_key(tail: dict[str, Any]) -> tuple[str, Any]:
#     value = tail["value"]
#     if tail["datatype"] == "wikibase-item":
#         value = value["id"]
#     else:
#         value = json.dumps(value, sort_keys=True, ensure_ascii=True)
#     return tail["datatype"], value


# def get_uniedit_pair_key(edit: dict[str, Any]) -> tuple[str, str]:
#     return edit["head_entity"]["id"], edit["property"]["id"]


# def get_uniedit_fact_key(edit: dict[str, Any]) -> tuple[Any, ...]:
#     return (*get_uniedit_pair_key(edit), *get_uniedit_tail_key(edit["tail_entity"]))


# def get_uniedit_tail_names(tail: dict[str, Any]) -> list[str | None]:
#     if tail["datatype"] != "wikibase-item":
#         return []
#     value = tail["value"]
#     return [value.get("label"), *(value.get("aliases") or [])]


# source_files = [
#     source_file
#     for split_files in uniedit_raw.values()
#     for source_file in split_files
# ]
# pair_objects = {}

# for source_file in tqdm(source_files, desc="Find multi-object edits"):
#     with open(source_file, encoding="utf-8") as file:
#         domain_records = json.load(file)
#     for record in domain_records.values():
#         edit = record["edit"]
#         pair_objects.setdefault(get_uniedit_pair_key(edit), set()).add(
#             get_uniedit_tail_key(edit["tail_entity"])
#         )

# multi_object_pairs = {
#     key for key, objects in pair_objects.items() if len(objects) > 1
# }
# uniedit_groups = {}
# excluded_records = 0

# for source_file in tqdm(source_files, desc="Merge UniEdit facts"):
#     with open(source_file, encoding="utf-8") as file:
#         domain_records = json.load(file)
#     for record in domain_records.values():
#         edit = record["edit"]
#         if get_uniedit_pair_key(edit) in multi_object_pairs:
#             excluded_records += 1
#             continue
#         key = get_uniedit_fact_key(edit)
#         if key not in uniedit_groups:
#             merged_edit = {
#                 key: value
#                 for key, value in edit.items()
#                 if key not in ("prompt", "target")
#             }
#             merged_edit["prompts"] = []
#             merged_edit["targets"] = []
#             uniedit_groups[key] = {
#                 "edit": merged_edit,
#                 "generality": [],
#                 "locality": [],
#             }
#         group = uniedit_groups[key]
#         extend_unique_uniedit_texts(group["edit"]["prompts"], [edit["prompt"]])
#         extend_unique_uniedit_texts(
#             group["edit"]["targets"],
#             [edit["target"], *get_uniedit_tail_names(edit["tail_entity"])],
#         )
#         group["generality"].extend(record["generality"].values())
#         group["locality"].extend(record["locality"].values())

# uniedit_merged = list(uniedit_groups.values())
# os.makedirs("../data/datasets/uniedit/merged", exist_ok=True)
# with open("../data/datasets/uniedit/merged/data.jsonl", "w", encoding="utf-8") as file:
#     for record in uniedit_merged:
#         file.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n")

# {
#     "multi_object_pairs": len(multi_object_pairs),
#     "excluded_records": excluded_records,
#     "merged_records": len(uniedit_merged),
# }

In [ ]:
# import json


# with open("../data/datasets/uniedit/merged/data.jsonl", encoding="utf-8") as file:
#     uniedit_merged = [json.loads(line) for line in file if line.strip()]
# len(uniedit_merged)

### Evidence Cleaning

In [ ]:
# import json
# import os
# import re
# import unicodedata
# from tqdm.auto import tqdm
# from typing import Any


# def normalize_uniedit_text(text) -> str:
#     return " ".join(str(text).split()).casefold()


# def extend_unique_uniedit_texts(output: list[str], values: list[Any]) -> None:
#     seen = {normalize_uniedit_text(value) for value in output}
#     for value in values:
#         if value is None or not str(value).strip():
#             continue
#         value = str(value).strip()
#         key = normalize_uniedit_text(value)
#         if key not in seen:
#             seen.add(key)
#             output.append(value)


# def get_uniedit_answer_targets(target: str, tail: dict[str, Any]) -> list[str]:
#     targets = []
#     tail_names = (
#         [
#             tail["value"].get("label"),
#             *(tail["value"].get("aliases") or []),
#         ]
#         if tail["datatype"] == "wikibase-item"
#         else []
#     )
#     extend_unique_uniedit_texts(targets, [target, *tail_names])
#     return targets


# UNIEDIT_ANSWER_ARTICLES = {"a", "an", "the"}


# def get_uniedit_words(text) -> tuple[str, ...]:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     return tuple(re.findall(r"\w+", text))


# def get_uniedit_answer_keys(text) -> set[str]:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     words = get_uniedit_words(text)
#     keys = {"".join(words)} if words else set()
#     compact = "".join(
#         character
#         for character in text
#         if character.isalnum() or unicodedata.category(character).startswith("S")
#     )
#     if compact:
#         keys.add(compact)
#     if len(words) > 1 and words[0] in UNIEDIT_ANSWER_ARTICLES:
#         keys.add("".join(words[1:]))
#     return keys


# def get_uniedit_entity_texts(entity: dict[str, Any]) -> list[str]:
#     return [
#         value
#         for value in [entity.get("label"), *(entity.get("aliases") or [])]
#         if value is not None and str(value).strip()
#     ]


# def get_uniedit_entity_answers(entity: dict[str, Any]) -> set[str]:
#     answers = set()
#     for value in get_uniedit_entity_texts(entity):
#         answers.update(get_uniedit_answer_keys(value))
#     return answers


# def contains_uniedit_text(text: str, value: str) -> bool:
#     text_words = get_uniedit_words(text)
#     value_words = get_uniedit_words(value)
#     return bool(value_words) and any(
#         text_words[index : index + len(value_words)] == value_words
#         for index in range(len(text_words) - len(value_words) + 1)
#     )


# def has_valid_uniedit_subject(hop: dict[str, Any]) -> bool:
#     return any(
#         contains_uniedit_text(hop["prompt"], value)
#         for value in get_uniedit_entity_texts(hop["head_entity"])
#     )


# def get_uniedit_tail_answers(tail: dict[str, Any]) -> set[str]:
#     value = tail["value"]
#     if tail["datatype"] == "wikibase-item":
#         return get_uniedit_entity_answers(value)
#     if isinstance(value, str):
#         return get_uniedit_answer_keys(value)
#     if isinstance(value, dict) and isinstance(value.get("text"), str):
#         return get_uniedit_answer_keys(value["text"])
#     return set()


# def has_valid_uniedit_target(
#     hop: dict[str, Any],
#     expected_targets: tuple[str, ...] = (),
# ) -> bool:
#     target_answers = get_uniedit_answer_keys(hop["target"])
#     tail_answers = get_uniedit_tail_answers(hop["tail_entity"])
#     if tail_answers:
#         for value in expected_targets:
#             if value is not None and str(value).strip():
#                 tail_answers.update(get_uniedit_answer_keys(value))
#         return bool(target_answers & tail_answers)
#     head_answers = get_uniedit_entity_answers(hop["head_entity"])
#     return not bool(target_answers & head_answers)


# def is_forward_uniedit_path(path: list[str]) -> bool:
#     return path[-2:] in (["f", "s"], ["f", "m"])


# def get_uniedit_path_containers(item: dict[str, Any]) -> list[dict[str, Any]]:
#     if item["path_type"] == "single":
#         return [item]
#     if item["path_type"] == "double":
#         return [item["single_path1"], item["single_path2"]]
#     raise ValueError(f"Unexpected path type: {item['path_type']}")


# def add_uniedit_evidence(
#     evidence: list[dict[str, Any]],
#     prompt: str,
#     targets: list[str],
# ) -> None:
#     key = normalize_uniedit_text(prompt)
#     for candidate in evidence:
#         if normalize_uniedit_text(candidate["prompt"]) == key:
#             extend_unique_uniedit_texts(candidate["targets"], targets)
#             return
#     evidence.append({"prompt": prompt, "targets": list(targets)})


# def get_uniedit_evidence(
#     item: dict[str, Any],
#     edit: dict[str, Any],
#     role: str,
#     edit_pair_keys: set[tuple[str, str]],
# ) -> list[dict[str, Any]]:
#     expected_path = (
#         ["h_e", "r_e", "t_e"]
#         if role == "generality"
#         else ["h_l", "r_l", "t_l"]
#     )
#     edit_fact_key = (
#         edit["head_entity"]["id"],
#         edit["property"]["id"],
#         edit["tail_entity"]["datatype"],
#         edit["tail_entity"]["value"].get("id")
#         if edit["tail_entity"]["datatype"] == "wikibase-item"
#         else json.dumps(edit["tail_entity"]["value"], sort_keys=True, ensure_ascii=True),
#     )
#     edit_head = edit["head_entity"]["id"]
#     edit_property = edit["property"]["id"]
#     evidence = []

#     for container in get_uniedit_path_containers(item):
#         for hop in container.get("one_hops", []):
#             if hop["path"] != expected_path:
#                 continue
#             if role == "generality":
#                 hop_fact_key = (
#                     hop["head_entity"]["id"],
#                     hop["property"]["id"],
#                     hop["tail_entity"]["datatype"],
#                     hop["tail_entity"]["value"].get("id")
#                     if hop["tail_entity"]["datatype"] == "wikibase-item"
#                     else json.dumps(hop["tail_entity"]["value"], sort_keys=True, ensure_ascii=True),
#                 )
#                 valid_hop = hop_fact_key == edit_fact_key
#             else:
#                 hop_pair_key = (
#                     hop["head_entity"]["id"],
#                     hop["property"]["id"],
#                 )
#                 valid_hop = (
#                     hop["property"]["id"] == edit_property
#                     and hop["head_entity"]["id"] != edit_head
#                     and hop_pair_key not in edit_pair_keys
#                 )
#             if not valid_hop:
#                 continue
#             expected_targets = edit["targets"] if role == "generality" else ()
#             if not has_valid_uniedit_subject(hop):
#                 continue
#             if not has_valid_uniedit_target(hop, expected_targets):
#                 continue
#             add_uniedit_evidence(
#                 evidence,
#                 hop["prompt"],
#                 get_uniedit_answer_targets(hop["target"], hop["tail_entity"]),
#             )

#     if (
#         item["path_type"] == "single"
#         and len(item["paths"]) == 1
#         and len(item["one_hops"]) == 1
#         and item["paths"][0][:3] == expected_path
#         and is_forward_uniedit_path(item["paths"][0])
#         and evidence
#     ):
#         hop_evidence = evidence
#         evidence = []
#         direct_targets = [item["target"]]
#         for candidate in hop_evidence:
#             extend_unique_uniedit_texts(direct_targets, candidate["targets"])
#         add_uniedit_evidence(evidence, item["prompt"], direct_targets)
#         for candidate in hop_evidence:
#             add_uniedit_evidence(
#                 evidence, candidate["prompt"], candidate["targets"]
#             )
#     return evidence


# def clean_uniedit_record(
#     record: dict[str, Any],
#     edit_pair_keys: set[tuple[str, str]],
# ) -> dict[str, Any]:
#     edit = record["edit"]
#     generality = []
#     for item in record["generality"]:
#         for evidence in get_uniedit_evidence(
#             item, edit, "generality", edit_pair_keys
#         ):
#             add_uniedit_evidence(
#                 generality, evidence["prompt"], evidence["targets"]
#             )
#     locality = []
#     for item in record["locality"]:
#         if item["loc_type"] != "property of edit":
#             continue
#         for evidence in get_uniedit_evidence(
#             item, edit, "locality", edit_pair_keys
#         ):
#             add_uniedit_evidence(
#                 locality, evidence["prompt"], evidence["targets"]
#             )
#     return {"edit": edit, "generality": generality, "locality": locality}


# uniedit_edit_pair_keys = {
#     (record["edit"]["head_entity"]["id"], record["edit"]["property"]["id"])
#     for record in uniedit_merged
# }
# uniedit_clean = [
#     clean_uniedit_record(record, uniedit_edit_pair_keys)
#     for record in tqdm(uniedit_merged, desc="Clean UniEdit facts")
# ]
# os.makedirs("../data/datasets/uniedit/clean", exist_ok=True)
# with open("../data/datasets/uniedit/clean/data.jsonl", "w", encoding="utf-8") as file:
#     for record in uniedit_clean:
#         file.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n")
# len(uniedit_clean)

In [ ]:
# import json


# with open("../data/datasets/uniedit/clean/data.jsonl", encoding="utf-8") as file:
#     uniedit_clean = [json.loads(line) for line in file if line.strip()]
# len(uniedit_clean)

### Final Dataset

In [ ]:
# import os
# import shutil
# from datasets import Dataset
# from typing import Any


# def normalize_uniedit_text(text) -> str:
#     return " ".join(str(text).split()).casefold()


# def extend_unique_uniedit_texts(output: list[str], values: list[Any]) -> None:
#     seen = {normalize_uniedit_text(value) for value in output}
#     for value in values:
#         if value is None or not str(value).strip():
#             continue
#         value = str(value).strip()
#         key = normalize_uniedit_text(value)
#         if key not in seen:
#             seen.add(key)
#             output.append(value)


# uniedit_rows = []
# for record in uniedit_clean:
#     edit_prompt = record["edit"]["prompts"][0]
#     edit_targets = list(record["edit"]["targets"])
#     for evidence in record["generality"]:
#         extend_unique_uniedit_texts(edit_targets, evidence["targets"])

#     seen_prompts = {normalize_uniedit_text(edit_prompt)}
#     generality = []
#     for prompt in [
#         *record["edit"]["prompts"][1:],
#         *(item["prompt"] for item in record["generality"]),
#     ]:
#         key = normalize_uniedit_text(prompt)
#         if key not in seen_prompts:
#             seen_prompts.add(key)
#             generality.append(prompt)

#     locality = []
#     for item in record["locality"]:
#         key = normalize_uniedit_text(item["prompt"])
#         if key in seen_prompts:
#             continue
#         seen_prompts.add(key)
#         locality.append({
#             "prompt": item["prompt"],
#             "targets": item["targets"],
#         })

#     if generality and locality:
#         uniedit_rows.append({
#             "edit": {"prompt": edit_prompt, "targets": edit_targets},
#             "generality": generality,
#             "locality": locality,
#         })

# def merge_uniedit_final_rows(left: dict[str, Any], right: dict[str, Any]) -> None:
#     extend_unique_uniedit_texts(left["edit"]["targets"], right["edit"]["targets"])
#     seen_prompts = {normalize_uniedit_text(left["edit"]["prompt"])}
#     generality = []
#     for prompt in [*left["generality"], *right["generality"]]:
#         key = normalize_uniedit_text(prompt)
#         if key not in seen_prompts:
#             seen_prompts.add(key)
#             generality.append(prompt)
#     locality = []
#     for item in [*left["locality"], *right["locality"]]:
#         key = normalize_uniedit_text(item["prompt"])
#         if key in seen_prompts:
#             for candidate in locality:
#                 if normalize_uniedit_text(candidate["prompt"]) == key:
#                     extend_unique_uniedit_texts(candidate["targets"], item["targets"])
#                     break
#             continue
#         seen_prompts.add(key)
#         locality.append({"prompt": item["prompt"], "targets": list(item["targets"])})
#     left["generality"] = generality
#     left["locality"] = locality
#
#
# rows_by_edit_prompt = {}
# ambiguous_edit_prompts = set()
# for row in uniedit_rows:
#     key = normalize_uniedit_text(row["edit"]["prompt"])
#     if key not in rows_by_edit_prompt:
#         rows_by_edit_prompt[key] = row
#         continue
#     existing_targets = {normalize_uniedit_text(value) for value in rows_by_edit_prompt[key]["edit"]["targets"]}
#     new_targets = {normalize_uniedit_text(value) for value in row["edit"]["targets"]}
#     if existing_targets & new_targets:
#         merge_uniedit_final_rows(rows_by_edit_prompt[key], row)
#     else:
#         ambiguous_edit_prompts.add(key)
#
# uniedit_rows = [
#     row
#     for key, row in rows_by_edit_prompt.items()
#     if key not in ambiguous_edit_prompts
# ]
# uniedit_final = Dataset.from_list(uniedit_rows)
# uniedit_final_path = "../data/datasets/uniedit/final"
# if os.path.exists(uniedit_final_path):
#     shutil.rmtree(uniedit_final_path)
# uniedit_final.save_to_disk(uniedit_final_path)

In [ ]:
# from datasets import load_from_disk


# uniedit = load_from_disk("../data/datasets/uniedit/final")
# uniedit

## Publishing

### Edit Success - Locality

In [ ]:
# from datasets import load_from_disk


# es_loc = load_from_disk("../data/datasets/counterfact/final")
# es_loc

In [ ]:
# es_loc.push_to_hub(
#     "Savassan/KnowledgeSurgeon",
#     config_name="es_loc",
#     split="data",
#     private=True,
# )

### Portability

In [ ]:
# from datasets import load_from_disk
#
#
# port = load_from_disk("../data/datasets/mquake-remastered/final")
# port

In [ ]:
# port.push_to_hub(
#     "Savassan/KnowledgeSurgeon",
#     config_name="port",
#     split="data",
#     private=True,
# )

### Similarity

In [ ]:
# from datasets import load_from_disk
#
#
# sim = load_from_disk("../data/datasets/uniedit/final")
# sim

In [ ]:
# sim.push_to_hub(
#     "Savassan/KnowledgeSurgeon",
#     config_name="similarity",
#     split="data",
#     private=True,
# )